# 02_SOLVER - KRONA LTP

Notebook operacional do Solver de Capacidade Finita do LTP.

Este arquivo contém somente implementação técnica, comentários de funcionamento e validações executáveis. Regras de negócio, histórico de decisões, dicionário de guias/colunas e legenda de status ficam centralizados no documento `00_MEMORIA_PROJETO_SOLVER_KRONA`.

## Fluxo técnico

1. Configuração de caminhos e parâmetros de execução.
2. Leitura do LTP e da estrutura de produtos.
3. Preparação da necessidade e explosão estrutural.
4. Montagem das alternativas produtivas.
5. Rateio com restrição global de recurso e ferramenta.
6. Fechamento iterativo pai/filho do M1.
7. Validações finais.
8. Exportação do Excel.
9. Encerramento do temporizador.


In [ ]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================


try:
    from functions import *
except Exception:
    import time
    class Temporizador:
        def iniciar(self):
            self._inicio = time.time()
            pass
        def finalizar(self):
            fim = time.time()
            inicio = getattr(self, '_inicio', fim)
            print(f'Tempo total: {fim - inicio:,.2f} segundos')
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
try:
    from IPython.display import display
except Exception:
    display = print
    
# Iniciando Temporizador
timer = Temporizador()
timer.iniciar()

# Versão do pipeline em teste
# A saída do solver sempre leva esta versão no nome do arquivo.
PIPELINE_VERSION = "v41"
OUTPUT_SOLVER_FILENAME = f"bd_SOLVER_{PIPELINE_VERSION}.xlsx"
ESTRUTURA_FILENAME = "estruturas.xlsx"

# Caminho padrão do projeto no Windows
OUTPUT_DIR_WINDOWS = Path(r"C:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT")
INPUT_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE_WINDOWS = OUTPUT_DIR_WINDOWS.parent / "01_INPUT" / ESTRUTURA_FILENAME

# Fallback para execução em ambiente local/sandbox
INPUT_FILE_FALLBACK = Path("/mnt/data/bd_LTP_NEC_SOLVER(5).xlsx") if Path("/mnt/data/bd_LTP_NEC_SOLVER(5).xlsx").exists() else (Path("/mnt/data/bd_LTP_NEC_SOLVER(4).xlsx") if Path("/mnt/data/bd_LTP_NEC_SOLVER(4).xlsx").exists() else Path("/mnt/data/bd_LTP_NEC_SOLVER.xlsx"))
OUTPUT_SOLVER_FILE_FALLBACK = Path("/mnt/data") / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE_FALLBACK = Path("/mnt/data/estruturas.xlsx") if Path("/mnt/data/estruturas.xlsx").exists() else Path("/mnt/data/estruturas(6).xlsx")

if INPUT_FILE_WINDOWS.exists():
    INPUT_FILE = INPUT_FILE_WINDOWS
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_WINDOWS
    ESTRUTURA_FILE = ESTRUTURA_FILE_WINDOWS if ESTRUTURA_FILE_WINDOWS.exists() else ESTRUTURA_FILE_FALLBACK
else:
    INPUT_FILE = INPUT_FILE_FALLBACK
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_FALLBACK
    ESTRUTURA_FILE = ESTRUTURA_FILE_FALLBACK


LOTE_MIN_FLAG = True
MULTIPLO_EMB_FLAG = True
TOL = 1e-7
DATA_HORA_EXECUCAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [ ]:

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def is_valid_alt(s):
    """Converte a coluna ALTERNATIVA_VALIDA para booleano de forma segura.

    Aceita booleanos reais e textos comuns como TRUE, SIM, YES, 1, VALIDA e ALTERNATIVA_VALIDA.
    Qualquer valor vazio ou diferente disso vira False.
    """
    if isinstance(s, pd.Series):
        if str(s.dtype).lower() == "bool":
            return s.fillna(False)
        txt = s.fillna(False).astype(str).str.strip().str.upper()
        return txt.isin(["TRUE", "VERDADEIRO", "SIM", "YES", "Y", "1", "VALIDA", "VÁLIDA", "ALTERNATIVA_VALIDA"])
    return bool(s)


def calcular_nec_pcs_se_necessario(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula NEC_PCS somente se a coluna não existir na base."""
    df = df.copy()
    if "NEC_PCS" in df.columns:
        df["NEC_PCS"] = to_num(df["NEC_PCS"])
        if "NEC_ESTRUTURA_PAI_PCS" not in df.columns:
            df["NEC_DIRETA_BRUTA_PCS"] = df["NEC_PCS"]
            df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
        else:
            df["NEC_ESTRUTURA_PAI_PCS"] = to_num(df["NEC_ESTRUTURA_PAI_PCS"])
        print("NEC_PCS lido da base de entrada.")
        return df

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    # Demanda bruta antes de deduzir estoque/ORI/transito.
    # Esta demanda é usada como base da explosão da estrutura, seguindo o comportamento do motor antigo
    # que explodia componentes antes de o corte/solver eliminar itens por estoque/capacidade.
    demanda_bruta = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
    )
    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")
    demanda_bruta = np.where(mesma_reg_nao, demanda_bruta + df["LTP_SALDO_PREV_PROX_MES_PCS"], demanda_bruta)
    demanda_bruta = pd.Series(demanda_bruta, index=df.index).clip(lower=0)
    demanda_bruta = demanda_bruta + df["LTP_COMP_NEC_PCS"]
    demanda_bruta = np.maximum(demanda_bruta, df["LIMIT_PCS"])

    # Demanda líquida para o solver: deduz estoque/transito/ORI/triangulação.
    base_comum = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"] - df["LTP_EST_TRANS_PCS"] - df["ORI_TOT_PCS"] - df["TRIANG_TOT_PCS"]
    )
    nec = np.where(mesma_reg_nao, base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"], base_comum)
    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    nec = np.maximum(nec, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (nec > 0) & (df["LOTE_MIN"] > 0)
        nec = np.where(mask_lote, np.maximum(nec, df["LOTE_MIN"]), nec)
    if MULTIPLO_EMB_FLAG:
        tipo = df.get("TIPO_PROD", "").astype(str).str.upper()
        mask_emb = (nec > 0) & (df["QTD_EMB"] > 0) & (tipo.isin(["PA", "MR"]))
        nec = np.where(mask_emb, np.ceil(nec / df["QTD_EMB"]) * df["QTD_EMB"], nec)

    df["NEC_PCS"] = pd.Series(nec, index=df.index).fillna(0)

    # Demanda bruta calculada para auditoria e para exceções de explosão estrutural.
    # Por padrão, a explosão usa a NEC líquida do pai (NEC_PCS).
    # Exceções de retorno de componente/transferência são ativadas depois que a estrutura é conhecida.
    nec_estrutura_bruta = demanda_bruta.copy()
    if LOTE_MIN_FLAG:
        mask_lote_estrutura = (nec_estrutura_bruta > 0) & (df["LOTE_MIN"] > 0)
        nec_estrutura_bruta = np.where(mask_lote_estrutura, np.maximum(nec_estrutura_bruta, df["LOTE_MIN"]), nec_estrutura_bruta)
    if MULTIPLO_EMB_FLAG:
        tipo = df.get("TIPO_PROD", "").astype(str).str.upper()
        mask_emb_estrutura = (nec_estrutura_bruta > 0) & (df["QTD_EMB"] > 0) & (tipo.isin(["PA", "MR"]))
        nec_estrutura_bruta = np.where(mask_emb_estrutura, np.ceil(nec_estrutura_bruta / df["QTD_EMB"]) * df["QTD_EMB"], nec_estrutura_bruta)
    df["NEC_DIRETA_BRUTA_PCS"] = pd.Series(nec_estrutura_bruta, index=df.index).fillna(0)
    df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
    print("NEC_PCS calculado internamente porque a base não trouxe a coluna.")
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()
    print("MES_REF processado:", mes_ref)
    print("Linhas bd_mes:", len(bd_mes))
    return bd_mes


def classificar_motivo_invalida(row):
    motivos = []
    if pd.isna(row.get("ALOC_REC")) or str(row.get("ALOC_REC", "")).strip() == "":
        motivos.append("ALOC_REC_AUSENTE")
    if pd.isna(row.get("COD_FER_UNID")) or str(row.get("COD_FER_UNID", "")).strip() == "":
        motivos.append("COD_FER_UNID_AUSENTE")
    if row.get("PCS_HORA", 0) <= TOL:
        motivos.append("PCS_HORA_ZERO_OU_INVALIDO")
    if row.get("HOR_REC", 0) <= TOL:
        motivos.append("HOR_REC_ZERO_OU_NEGATIVO")
    if row.get("HOR_FER", 0) <= TOL:
        motivos.append("HOR_FER_ZERO_OU_NEGATIVO")
    if row.get("HOR_CAP", 0) <= TOL:
        motivos.append("HOR_CAP_ZERO_OU_NEGATIVO")
    return " | ".join(dict.fromkeys(motivos)) if motivos else "ALTERNATIVA_VALIDA"


def formatar_excel(caminho_arquivo):
    wb = load_workbook(caminho_arquivo)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E2F3")
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = Border(bottom=thin_gray)
        for col_idx, column_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in column_cells:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 45)
        ws.auto_filter.ref = ws.dimensions
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
    wb.save(caminho_arquivo)



# ============================================================
# 01B. FUNÇÕES - EXPLOSÃO DE ESTRUTURA COM ESTOQUE
# ============================================================

from collections import defaultdict, deque


def normalizar_codigo_serie(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )


def ler_estrutura_produto(caminho_estrutura: Path) -> pd.DataFrame:
    if not caminho_estrutura.exists():
        print("Arquivo de estrutura não encontrado. A explosão será ignorada:", caminho_estrutura)
        return pd.DataFrame()

    estrutura = pd.read_excel(
        caminho_estrutura,
        sheet_name="estruturas",
        dtype={"cod_prod_acabado": str, "cod_insumo": str, "empresa": str}
    )

    required = ["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]
    missing = [c for c in required if c not in estrutura.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na estrutura: {missing}")

    estrutura = estrutura.copy()
    estrutura["empresa"] = estrutura["empresa"].astype("string").fillna("").str.strip()
    estrutura["cod_prod_acabado"] = normalizar_codigo_serie(estrutura["cod_prod_acabado"])
    estrutura["cod_insumo"] = normalizar_codigo_serie(estrutura["cod_insumo"])
    estrutura["qtd_utilizada_pcs"] = to_num(estrutura["qtd_utilizada_pcs"])
    estrutura = estrutura[
        (estrutura["empresa"] != "")
        & (estrutura["cod_prod_acabado"] != "")
        & (estrutura["cod_insumo"] != "")
    ].copy()

    estrutura = estrutura.drop_duplicates(["empresa", "cod_prod_acabado", "cod_insumo"], keep="first")
    print("Estrutura lida:", estrutura.shape)
    return estrutura


def calcular_estoque_componentes_pos_demanda_direta(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula estoque disponível para a explosão após consumir a demanda direta/bruta do próprio item.

    Mantém a lógica do pipeline antigo: estoque total = estoque inicial + trânsito + ORI + triangulação.
    Depois abate a demanda bruta do item, sem descontar estoque nessa demanda, para evitar usar na estrutura
    o estoque que já estaria comprometido com a necessidade direta do próprio produto.
    """
    df = df.copy()
    required = [
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_EST_SEG_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    df["COD_PROD"] = normalizar_codigo_serie(df["COD_PROD"])
    df["UNID_PROD"] = df["UNID_PROD"].astype("string").fillna("").str.strip()
    if "UNID_FAT" not in df.columns:
        df["UNID_FAT"] = df["UNID_PROD"]
    df["UNID_FAT"] = df["UNID_FAT"].astype("string").fillna("").str.strip()
    tipo_prod = df.get("TIPO_PROD", "").astype(str).str.upper()
    mesma_reg = df.get("MESMA_REG", "SIM").astype(str).str.upper()

    estoque_total = (
        df["LTP_EST_INI_PCS"]
        + df["LTP_EST_TRANS_PCS"]
        + df["ORI_TOT_PCS"]
        + df["TRIANG_TOT_PCS"]
    )

    demanda_base = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
    )
    mask_mesma_reg_nao = mesma_reg.eq("NAO")
    demanda_base = demanda_base.where(~mask_mesma_reg_nao, demanda_base + df["LTP_SALDO_PREV_PROX_MES_PCS"])

    demanda_bruta = np.maximum(demanda_base, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (demanda_bruta > 0) & (df["LOTE_MIN"] > 0)
        demanda_bruta = np.where(mask_lote, np.maximum(demanda_bruta, df["LOTE_MIN"]), demanda_bruta)

    if MULTIPLO_EMB_FLAG:
        mask_emb = (demanda_bruta > 0) & (df["QTD_EMB"] > 0) & (tipo_prod.isin(["PA", "MR"]))
        demanda_bruta = np.where(mask_emb, np.ceil(demanda_bruta / df["QTD_EMB"]) * df["QTD_EMB"], demanda_bruta)

    df["ESTOQUE_TOTAL_PCS"] = estoque_total
    df["DEMANDA_BRUTA_DIRETA_PCS"] = np.maximum(demanda_bruta, 0)
    df["ESTOQUE_DISP_EXPLOSAO_PCS"] = np.maximum(df["ESTOQUE_TOTAL_PCS"] - df["DEMANDA_BRUTA_DIRETA_PCS"], 0)

    # Regra herdada do motor de cortes:
    # o estoque disponível para a explosão é localizado pela unidade de faturamento/destino.
    # Isso permite que uma demanda Nordeste do pai gere necessidade em uma linha do componente
    # produzida em Matriz, desde que a linha do componente tenha UNID_FAT = Nordeste.
    estoque = (
        df.groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(
            ESTOQUE_TOTAL_PCS=("ESTOQUE_TOTAL_PCS", "max"),
            DEMANDA_BRUTA_DIRETA_PCS=("DEMANDA_BRUTA_DIRETA_PCS", "max"),
            ESTOQUE_DISP_EXPLOSAO_PCS=("ESTOQUE_DISP_EXPLOSAO_PCS", "max")
        )
        .rename(columns={"UNID_FAT": "UNID_PROD"})
    )
    return estoque


def explodir_estrutura_multinivel_com_estoque(
    bd_estrutura: pd.DataFrame,
    bd_demanda_raiz: pd.DataFrame,
    bd_estoque_componentes: pd.DataFrame,
    bd_mapa_componentes: pd.DataFrame = None,
    max_nivel: int = 30
):
    cols = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM",
        "NEC_RAIZ_PCS", "COD_PROD_ACAB", "COD_INSUMO", "QTD_UTIL_PCS",
        "NEC_GERADA_BRUTA_PCS", "ESTOQUE_COMP_ANTES_PCS", "ESTOQUE_CONSUMIDO_PCS",
        "NEC_GERADA_LIQ_PCS", "ESTOQUE_COMP_DEPOIS_PCS", "NIVEL", "TRILHA"
    ]
    if bd_estrutura.empty or bd_demanda_raiz.empty:
        return pd.DataFrame(columns=cols), pd.DataFrame(columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])

    estrutura_idx = defaultdict(lambda: defaultdict(list))
    for emp, pai, filho, qtd in bd_estrutura[["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]].itertuples(index=False, name=None):
        estrutura_idx[str(emp)][str(pai)].append((str(filho), float(qtd or 0)))

    estoque_dict = {
        (str(unid), str(cod)): float(est or 0)
        for cod, unid, est in bd_estoque_componentes[["COD_PROD", "UNID_PROD", "ESTOQUE_DISP_EXPLOSAO_PCS"]].itertuples(index=False, name=None)
    }

    # : regra geral de retorno da necessidade estrutural.
    # A estrutura é localizada pela unidade produtiva do pai.
    # A necessidade do componente retorna pela unidade de faturamento da origem quando existir
    # linha do componente nessa unidade; se não existir, usa a unidade produtiva/estrutural
    # da origem como fallback. Isso preserva a unidade que gerou a necessidade sem criar colunas novas.
    if bd_mapa_componentes is None or bd_mapa_componentes.empty:
        mapa_comp = pd.DataFrame(columns=["COD_PROD", "UNID_FAT", "UNID_PROD"])
    else:
        mapa_comp = bd_mapa_componentes.copy()
    for _c in ["COD_PROD", "UNID_FAT", "UNID_PROD"]:
        if _c not in mapa_comp.columns:
            mapa_comp[_c] = ""
        mapa_comp[_c] = mapa_comp[_c].astype("string").fillna("").str.strip()
    mapa_comp["COD_PROD"] = normalizar_codigo_serie(mapa_comp["COD_PROD"])
    mapa_comp = mapa_comp.drop_duplicates(["COD_PROD", "UNID_FAT"], keep="first")
    comp_por_fat = {
        (str(cod), str(unid_fat)): str(unid_prod)
        for cod, unid_fat, unid_prod in mapa_comp[["COD_PROD", "UNID_FAT", "UNID_PROD"]].itertuples(index=False, name=None)
    }

    def resolver_unidade_retorno_componente(cod_filho, unid_fat_origem, unid_estrutura_origem):
        cod_filho = str(cod_filho)
        unid_fat_origem = str(unid_fat_origem or "").strip()
        unid_estrutura_origem = str(unid_estrutura_origem or "").strip()
        # 1) regra operacional para famílias de Acessórios: quando a estrutura técnica é de Acessórios,
        #    a necessidade do componente deve voltar para Acessórios, mesmo que a demanda original esteja
        #    em Matriz/Nordeste. Esse é o caso validado do 0860TB-BR.
        if unid_estrutura_origem == "Acessórios" and (cod_filho, unid_estrutura_origem) in comp_por_fat:
            return unid_estrutura_origem, comp_por_fat[(cod_filho, unid_estrutura_origem)]
        # 2) regra principal geral: preservar a UNID_FAT que gerou a necessidade.
        if (cod_filho, unid_fat_origem) in comp_por_fat:
            return unid_fat_origem, comp_por_fat[(cod_filho, unid_fat_origem)]
        # 3) fallback: quando o componente não existe na UNID_FAT da origem,
        #    usar a unidade produtiva/estrutura da origem se ela existir como UNID_FAT do componente.
        if (cod_filho, unid_estrutura_origem) in comp_por_fat:
            return unid_estrutura_origem, comp_por_fat[(cod_filho, unid_estrutura_origem)]
        # 4) fallback conservador: mantém a UNID_FAT da origem e segue explodindo na unidade estrutural atual.
        return unid_fat_origem if unid_fat_origem else unid_estrutura_origem, unid_estrutura_origem

    resultados = []
    problemas = []

    ordem_cols = ["MES_REF", "UNID_PROD", "COD_PROD", "ID_PROD_UNID_FAT"]
    bd_demanda_raiz_ord = bd_demanda_raiz.sort_values([c for c in ordem_cols if c in bd_demanda_raiz.columns]).copy()

    for row in bd_demanda_raiz_ord[["MES_REF", "UNID_PROD", "UNID_FAT_ORIGEM", "ID_PROD_UNID_FAT", "COD_PROD", "NEC_PCS"]].itertuples(index=False, name=None):
        mes_ref, unid_estrutura, unid_fat_origem, id_origem, cod_origem, nec_raiz = row
        emp = str(unid_estrutura)
        unid_fat_origem = str(unid_fat_origem or emp).strip()
        root = str(cod_origem)
        nec_raiz = float(nec_raiz or 0)
        if nec_raiz <= TOL:
            continue
        estrutura_empresa = estrutura_idx.get(emp, {})
        if root not in estrutura_empresa:
            continue

        # Na fila entra a necessidade líquida do pai.
        # A fila carrega a unidade estrutural do nó para localizar seus filhos, mantendo separada a UNID_FAT da origem.
        fila = deque([(emp, root, 0, nec_raiz, root, {root})])
        while fila:
            emp_no, pai, nivel, nec_pai_liq, trilha, caminho = fila.popleft()
            if nivel >= max_nivel:
                problemas.append((emp_no, root, trilha, "MAX_NIVEL_ATINGIDO"))
                continue

            estrutura_empresa_no = estrutura_idx.get(str(emp_no), {})
            for filho, qtd_util in estrutura_empresa_no.get(pai, []):
                nivel_filho = nivel + 1
                trilha_filho = f"{trilha} -> {filho}"
                ciclo = filho in caminho
                nec_bruta = float(nec_pai_liq or 0) * float(qtd_util or 0)

                if ciclo:
                    problemas.append((emp_no, root, trilha_filho, "CICLO_ESTRUTURA"))
                    continue
                if nec_bruta <= TOL:
                    continue

                unid_retorno, unid_prod_componente = resolver_unidade_retorno_componente(filho, unid_fat_origem, emp_no)
                chave_estoque = (unid_retorno, str(filho))
                estoque_antes = float(estoque_dict.get(chave_estoque, 0.0))
                estoque_consumido = min(estoque_antes, nec_bruta)
                nec_liq = max(nec_bruta - estoque_consumido, 0.0)
                estoque_depois = max(estoque_antes - estoque_consumido, 0.0)
                estoque_dict[chave_estoque] = estoque_depois

                resultados.append((
                    mes_ref, unid_retorno, id_origem, root, nec_raiz, pai, filho, qtd_util,
                    nec_bruta, estoque_antes, estoque_consumido, nec_liq, estoque_depois,
                    nivel_filho, trilha_filho
                ))

                # Só explode abaixo do componente se ainda existe necessidade líquida dele.
                # Para localizar os filhos do componente, usa a unidade produtiva do componente.
                if nec_liq > TOL:
                    fila.append((unid_prod_componente, filho, nivel_filho, nec_liq, trilha_filho, caminho | {filho}))

    bd_explodida = pd.DataFrame.from_records(resultados, columns=cols)
    bd_problemas = pd.DataFrame.from_records(problemas, columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])
    return bd_explodida, bd_problemas

def consolidar_necessidade_com_estrutura(bd_mes: pd.DataFrame, bd_estrutura_explodida: pd.DataFrame):
    """Consolida demanda direta e demanda de componente seguindo o motor antigo.

    Regra principal :
    - a explosão da estrutura nasce pela unidade produtiva do pai;
    - a necessidade líquida do componente volta para o LTP pela unidade de faturamento da origem quando existir linha do componente nessa unidade;
    - quando não existir essa linha, usa a unidade produtiva/estrutural da origem como fallback;
    - a chave de retorno continua sem alterar layout:
        componente.COD_PROD = COD_INSUMO explodido
        componente.UNID_FAT = UNID_PROD registrada na explosão
    - o solver passa a trabalhar com duas naturezas de demanda:
        NEC_DIRETA_PCS = demanda própria do item;
        NEC_COMP_PCS   = demanda líquida gerada por pais/componentes;
        NEC_PCS        = NEC_DIRETA_PCS + NEC_COMP_PCS.
    """
    bd_mes_out = bd_mes.copy()
    if "UNID_FAT" not in bd_mes_out.columns:
        bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_PROD"]
    bd_mes_out["COD_PROD"] = normalizar_codigo_serie(bd_mes_out["COD_PROD"])
    bd_mes_out["UNID_PROD"] = bd_mes_out["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out["NEC_PCS_ORIGINAL"] = to_num(bd_mes_out["NEC_PCS"])
    bd_mes_out["NEC_DIRETA_PCS"] = bd_mes_out["NEC_PCS_ORIGINAL"]

    if bd_estrutura_explodida is None or bd_estrutura_explodida.empty:
        nec_comp_retorno = pd.DataFrame(columns=[
            "MES_REF", "UNID_FAT", "COD_PROD",
            "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"
        ])
    else:
        nec_comp_retorno = (
            bd_estrutura_explodida
            .groupby(["MES_REF", "UNID_PROD", "COD_INSUMO"], as_index=False)
            .agg(
                NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
                ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
                NEC_COMP_PCS=("NEC_GERADA_LIQ_PCS", "sum")
            )
            .rename(columns={"UNID_PROD": "UNID_FAT", "COD_INSUMO": "COD_PROD"})
        )
        nec_comp_retorno["COD_PROD"] = normalizar_codigo_serie(nec_comp_retorno["COD_PROD"])
        nec_comp_retorno["UNID_FAT"] = nec_comp_retorno["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out = bd_mes_out.merge(
        nec_comp_retorno,
        on=["MES_REF", "UNID_FAT", "COD_PROD"],
        how="left"
    )

    for c in ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"]:
        bd_mes_out[c] = to_num(bd_mes_out[c])

    # Alias mantido para compatibilidade com guias antigas.
    bd_mes_out["NEC_EXPLODIDA_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["LTP_COMP_NEC_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = bd_mes_out["NEC_DIRETA_PCS"] + bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["NEC_PCS"] = bd_mes_out["NEC_PCS_CONSOLIDADA"]

    chave_item = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT"]
    extras = ["DESC_PROD", "TIPO_PROD"]
    agg = {
        "NEC_DIRETA_PCS": "max",
        "NEC_COMP_PCS": "max",
        "NEC_EXPLODIDA_BRUTA_PCS": "max",
        "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS": "max",
        "NEC_EXPLODIDA_PCS": "max",
        "NEC_PCS_CONSOLIDADA": "max",
    }
    for c in extras:
        if c in bd_mes_out.columns:
            agg[c] = "first"
    nec_consolidada = (
        bd_mes_out.groupby([c for c in chave_item if c in bd_mes_out.columns], as_index=False)
        .agg(agg)
    )
    nec_consolidada["TEM_ROTEIRO_NO_SOLVER"] = nec_consolidada["ID_PROD_UNID_FAT"].notna()

    return bd_mes_out, nec_consolidada

# ============================================================
# 01C. FUNÇÕES - AJUSTE PÓS-SOLVER PAI/FILHO -  MOTIVO AJUSTADO
# ============================================================

def calcular_ajuste_pos_solver_pai_filho(estrutura_explodida: pd.DataFrame, nec_consolidada_solver: pd.DataFrame, nao_atend_item: pd.DataFrame, plano_producao: pd.DataFrame):
    """Calcula diagnóstico pós-solver de viabilidade pai/filho.

    :
    - o filho/componente sustenta o pai com estoque já consumido na explosão + produção do filho pelo solver;
    - a produção do filho continua distribuída proporcionalmente entre demanda direta e estrutural quando houver necessidade total compartilhada;
    - todo corte do pai recalcula a capacidade liberada para realocação posterior.
    """
    cols_ajuste = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "COD_PROD_ACAB", "COD_INSUMO", "NIVEL", "TRILHA", "QTD_UTIL_PCS",
        "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "FATOR_ATEND_PAI_SOLVER", "NEC_GERADA_LIQ_PCS", "ESTOQUE_CONSUMIDO_PCS", "NEC_FILHO_POS_CORTE_PAI_PCS",
        "NEC_DIRETA_FILHO_PCS", "NEC_TOTAL_FILHO_SOLVER_PCS", "QTD_FILHO_SOLVER_TOTAL", "QTD_FILHO_SOLVER_ESTRUTURA_PCS", "QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS",
        "TEM_ROTEIRO_FILHO_NO_SOLVER", "FATOR_ATEND_FILHO_TOTAL", "FATOR_FILHO_USADO_PARA_LIMITAR", "NEC_FILHO_ATENDIDA_ESTRUTURA_PCS",
        "QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    cols_resumo = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "QTD_PAI_FINAL_VIAVEL_EST",
        "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_ATEND_PAI_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_COMPONENTES_ANALISADOS",
        "QTD_COMPONENTES_LIMITANTES", "QTD_COMPONENTES_SEM_ROTEIRO", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    if estrutura_explodida is None or estrutura_explodida.empty:
        return pd.DataFrame(columns=cols_ajuste), pd.DataFrame(columns=cols_resumo), pd.DataFrame()

    ajuste = estrutura_explodida.copy()
    for c in ["NEC_RAIZ_PCS", "NEC_GERADA_LIQ_PCS", "QTD_UTIL_PCS", "ESTOQUE_CONSUMIDO_PCS"]:
        if c not in ajuste.columns:
            ajuste[c] = 0.0
        ajuste[c] = to_num(ajuste[c])

    cols_atend_base = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT", "QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]
    cols_atend_base = [c for c in cols_atend_base if c in nao_atend_item.columns]
    atend = nao_atend_item[cols_atend_base].copy()
    for c in ["QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]:
        if c not in atend.columns:
            atend[c] = 0.0
        atend[c] = to_num(atend[c])

    if nec_consolidada_solver is not None and not nec_consolidada_solver.empty and "TEM_ROTEIRO_NO_SOLVER" in nec_consolidada_solver.columns:
        roteiro_filho = nec_consolidada_solver[["MES_REF", "COD_PROD", "UNID_FAT", "TEM_ROTEIRO_NO_SOLVER"]].copy()
        roteiro_filho = roteiro_filho.rename(columns={"COD_PROD": "COD_INSUMO", "UNID_FAT": "UNID_PROD", "TEM_ROTEIRO_NO_SOLVER": "TEM_ROTEIRO_FILHO_NO_SOLVER"})
    else:
        roteiro_filho = pd.DataFrame(columns=["MES_REF", "COD_INSUMO", "UNID_PROD", "TEM_ROTEIRO_FILHO_NO_SOLVER"])

    atend_pai = atend.rename(columns={
        "ID_PROD_UNID_FAT": "ID_PROD_UNID_FAT_ORIGEM",
        "QTD_ATENDIDA_SOLVER": "QTD_PAI_SOLVER",
        "NEC_PCS": "NEC_PAI_SOLVER_PCS"
    })[["MES_REF", "ID_PROD_UNID_FAT_ORIGEM", "QTD_PAI_SOLVER", "NEC_PAI_SOLVER_PCS"]]
    ajuste = ajuste.merge(atend_pai, on=["MES_REF", "ID_PROD_UNID_FAT_ORIGEM"], how="left")
    ajuste["QTD_PAI_SOLVER"] = ajuste["QTD_PAI_SOLVER"].fillna(0)
    ajuste["FATOR_ATEND_PAI_SOLVER"] = np.where(ajuste["NEC_RAIZ_PCS"] > TOL, (ajuste["QTD_PAI_SOLVER"] / ajuste["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] = ajuste["NEC_GERADA_LIQ_PCS"] * ajuste["FATOR_ATEND_PAI_SOLVER"]

    atend_filho = atend.rename(columns={
        "COD_PROD": "COD_INSUMO",
        "UNID_FAT": "UNID_PROD_ESTRUTURA",
        "QTD_ATENDIDA_SOLVER": "QTD_FILHO_SOLVER_TOTAL",
        "NEC_PCS": "NEC_TOTAL_FILHO_SOLVER_PCS",
        "NEC_DIRETA_PCS": "NEC_DIRETA_FILHO_PCS"
    })[["MES_REF", "UNID_PROD_ESTRUTURA", "COD_INSUMO", "QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]]
    atend_filho = atend_filho.rename(columns={"UNID_PROD_ESTRUTURA": "UNID_PROD"})
    ajuste = ajuste.merge(atend_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    ajuste = ajuste.merge(roteiro_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    for c in ["QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]:
        ajuste[c] = ajuste[c].fillna(0)
    ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"] = ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"].fillna(False).astype(bool)

    # Atendimento total do filho no solver e parcela da produção do filho atribuída à necessidade estrutural.
    ajuste["FATOR_ATEND_FILHO_TOTAL"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["QTD_FILHO_SOLVER_TOTAL"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        1.0
    )
    ajuste["PESO_ESTRUTURA_NO_FILHO"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        0.0
    )
    ajuste["QTD_FILHO_SOLVER_ESTRUTURA_PCS"] = ajuste["QTD_FILHO_SOLVER_TOTAL"] * ajuste["PESO_ESTRUTURA_NO_FILHO"]

    # Regra : componente sustenta o pai com estoque já consumido na explosão + produção do filho alocada à estrutura.
    ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"] = ajuste["ESTOQUE_CONSUMIDO_PCS"].fillna(0) + ajuste["QTD_FILHO_SOLVER_ESTRUTURA_PCS"].fillna(0)
    ajuste["NEC_FILHO_ATENDIDA_ESTRUTURA_PCS"] = np.minimum(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"], ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"])

    ajuste["QTD_PAI_SUSTENTADA_PELO_FILHO"] = np.where(
        ajuste["QTD_UTIL_PCS"] > TOL,
        ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"] / ajuste["QTD_UTIL_PCS"],
        ajuste["QTD_PAI_SOLVER"]
    )
    ajuste["QTD_PAI_FINAL_LINHA"] = ajuste["QTD_PAI_SOLVER"]
    mask_limita = (ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])
    ajuste.loc[mask_limita, "QTD_PAI_FINAL_LINHA"] = np.minimum(
        ajuste.loc[mask_limita, "QTD_PAI_SOLVER"],
        ajuste.loc[mask_limita, "QTD_PAI_SUSTENTADA_PELO_FILHO"]
    )
    ajuste["FATOR_FILHO_USADO_PARA_LIMITAR"] = np.where(
        ajuste["QTD_PAI_SOLVER"] > TOL,
        (ajuste["QTD_PAI_FINAL_LINHA"] / ajuste["QTD_PAI_SOLVER"]).clip(0, 1),
        1.0
    )
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0
    ajuste.loc[~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"], "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0

    ch_origem = ["MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM"]
    resumo = ajuste.groupby(ch_origem, as_index=False).agg(
        NEC_RAIZ_PCS=("NEC_RAIZ_PCS", "max"),
        QTD_PAI_SOLVER=("QTD_PAI_SOLVER", "max"),
        FATOR_ATEND_PAI_SOLVER=("FATOR_ATEND_PAI_SOLVER", "max"),
        QTD_PAI_FINAL_VIAVEL_EST=("QTD_PAI_FINAL_LINHA", "min"),
        QTD_COMPONENTES_ANALISADOS=("COD_INSUMO", "nunique")
    )
    resumo["QTD_PAI_FINAL_VIAVEL_EST"] = np.minimum(resumo["QTD_PAI_FINAL_VIAVEL_EST"], resumo["QTD_PAI_SOLVER"])
    resumo["QTD_REDUCAO_POS_PAI_FILHO"] = (resumo["QTD_PAI_SOLVER"] - resumo["QTD_PAI_FINAL_VIAVEL_EST"]).clip(lower=0)
    resumo["FATOR_FINAL_VIAVEL_ORIGEM"] = np.where(resumo["NEC_RAIZ_PCS"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    resumo["FATOR_AJUSTE_SOBRE_SOLVER"] = np.where(resumo["QTD_PAI_SOLVER"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["QTD_PAI_SOLVER"]).clip(0, 1), 0)

    limitantes = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"]) & (ajuste["QTD_PAI_FINAL_LINHA"] < ajuste["QTD_PAI_SOLVER"] - TOL)]
    qtd_limitantes = limitantes.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_LIMITANTES") if not limitantes.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_LIMITANTES"])
    sem_roteiro = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])]
    qtd_sem_roteiro = sem_roteiro.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_SEM_ROTEIRO") if not sem_roteiro.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_SEM_ROTEIRO"])
    resumo = resumo.merge(qtd_limitantes, on=ch_origem, how="left").merge(qtd_sem_roteiro, on=ch_origem, how="left")
    resumo["QTD_COMPONENTES_LIMITANTES"] = resumo["QTD_COMPONENTES_LIMITANTES"].fillna(0).astype(int)
    resumo["QTD_COMPONENTES_SEM_ROTEIRO"] = resumo["QTD_COMPONENTES_SEM_ROTEIRO"].fillna(0).astype(int)
    resumo["MOTIVO_AJUSTE_PAI_FILHO"] = np.select(
        [resumo["QTD_PAI_SOLVER"] <= TOL, resumo["FATOR_ATEND_PAI_SOLVER"] < 1 - TOL, resumo["QTD_COMPONENTES_LIMITANTES"] > 0, resumo["QTD_COMPONENTES_SEM_ROTEIRO"] > 0],
        ["PAI_NAO_PRODUZIDO_PELO_SOLVER", "PAI_CORTADO_PELO_SOLVER", "FILHO_COM_ROTEIRO_LIMITOU_PAI", "FILHO_SEM_ROTEIRO_EXPOSTO_NAO_LIMITA"],
        default="SEM_AJUSTE_POS_SOLVER"
    )

    ajuste = ajuste.merge(resumo[ch_origem + ["QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"]], on=ch_origem, how="left")
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_NECESSIDADE_FILHO_POS_CORTE_PAI"

    ajuste = ajuste[[c for c in cols_ajuste if c in ajuste.columns]].copy()
    resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].copy()

    fatores = resumo.rename(columns={"ID_PROD_UNID_FAT_ORIGEM": "ID_PROD_UNID_FAT"})[["MES_REF", "ID_PROD_UNID_FAT", "FATOR_AJUSTE_SOBRE_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM"]]
    if plano_producao is not None and not plano_producao.empty:
        cap_lib = plano_producao.merge(fatores, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
        cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"] = cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"].fillna(1.0)
        cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"] = cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"].fillna(1.0)
        for c in ["QTD_PRODUZIR_SOLVER", "PCS_HORA", "HR_PRODUZIR_SOLVER"]:
            cap_lib[c] = to_num(cap_lib[c]) if c in cap_lib.columns else 0.0
        cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = cap_lib["QTD_PRODUZIR_SOLVER"] * cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"]
        cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] = (cap_lib["QTD_PRODUZIR_SOLVER"] - cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"]).clip(lower=0)
        cap_lib["HR_LIBERADA_EST"] = np.where(cap_lib["PCS_HORA"] > TOL, cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] / cap_lib["PCS_HORA"], 0)
        cap_lib = cap_lib[cap_lib["HR_LIBERADA_EST"] > TOL].copy()
        cap_lib = cap_lib[[c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "QTD_PRODUZIR_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_PRODUZIR_SOLVER", "HR_LIBERADA_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if c in cap_lib.columns]].copy()
    else:
        cap_lib = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "ALOC_REC", "COD_FER_UNID", "HR_LIBERADA_EST"])

    return ajuste, resumo, cap_lib


In [ ]:

# ============================================================
# 02. LEITURA DA BASE + EXPLOSÃO INICIAL LÍQUIDA DA ESTRUTURA
# ============================================================

bd = pd.read_excel(INPUT_FILE, dtype={"COD_PROD": str, "ID_PROD_UNID_FAT": str, "UNID_PROD": str, "UNID_FAT": str})
print("Base lida:", bd.shape)
print("Colunas:", len(bd.columns))

bd["COD_PROD"] = normalizar_codigo_serie(bd["COD_PROD"])
bd["UNID_PROD"] = bd["UNID_PROD"].astype("string").fillna("").str.strip()
if "UNID_FAT" in bd.columns:
    bd["UNID_FAT"] = bd["UNID_FAT"].astype("string").fillna("").str.strip()

bd = calcular_nec_pcs_se_necessario(bd)
bd_mes = preparar_base(bd)

estrutura_produto = ler_estrutura_produto(ESTRUTURA_FILE)
estoque_componentes = calcular_estoque_componentes_pos_demanda_direta(bd_mes)

# : a estrutura é localizada pela unidade produtiva do pai.
# A UNID_FAT da origem é preservada separadamente para devolver a necessidade do componente ao plano.
bd_mes["UNID_ESTRUTURA"] = bd_mes["UNID_PROD"].astype("string").fillna("").str.strip()
bd_mes["UNID_FAT_ORIGEM"] = bd_mes["UNID_FAT"].where(
    bd_mes["UNID_FAT"].astype("string").fillna("").str.strip().ne(""),
    bd_mes["UNID_PROD"]
).astype("string").fillna("").str.strip()

# : a raiz da explosão também deve respeitar estoque/cobertura antes de gerar filhos.
# Regra vigente: a explosão usa somente NEC_ESTRUTURA_PAI_PCS líquida.
# Se o pai raiz tem demanda bruta, mas estoque/trânsito cobre 100%, ele fica no plano com produção zero
# e MOTIVO_STATUS = DEMANDA_DIRETA_COBERTA_POR_ESTOQUE, mas não gera necessidade produtiva para componentes.
mask_pai_coberto_sem_explodir = (
    (to_num(bd_mes["NEC_ESTRUTURA_PAI_PCS"]) <= TOL)
    & (to_num(bd_mes.get("NEC_DIRETA_BRUTA_PCS", 0)) > TOL)
)
print(":", int(mask_pai_coberto_sem_explodir.sum()))

demanda_raiz_estrutura = (
    bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD"], as_index=False)
    .agg(
        UNID_PROD=("UNID_ESTRUTURA", "first"),
        UNID_FAT_ORIGEM=("UNID_FAT_ORIGEM", "first"),
        NEC_PCS=("NEC_ESTRUTURA_PAI_PCS", "max")
    )
)
demanda_raiz_estrutura = demanda_raiz_estrutura[demanda_raiz_estrutura["NEC_PCS"] > TOL].copy()

mapa_componentes_retorno = (
    bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["COD_PROD", "UNID_FAT"], as_index=False)
    .agg(UNID_PROD=("UNID_PROD", "first"))
)

estrutura_explodida, problemas_estrutura = explodir_estrutura_multinivel_com_estoque(
    estrutura_produto,
    demanda_raiz_estrutura,
    estoque_componentes,
    mapa_componentes_retorno,
    max_nivel=30
)

bd_mes, nec_consolidada_solver = consolidar_necessidade_com_estrutura(bd_mes, estrutura_explodida)

# Validação direcionada  do caso crítico 0544 -> 0544NE.
try:
    _estr_0544 = estrutura_explodida[(estrutura_explodida["COD_PROD_ORIGEM"].astype(str) == "0544") & (estrutura_explodida["COD_INSUMO"].astype(str) == "0544NE")].copy()
    _estr_0544ne = nec_consolidada_solver[nec_consolidada_solver["COD_PROD"].astype(str).eq("0544NE")].copy()
    print("VALIDACAO_V23_0544_0544NE_ESTRUTURA_LINHAS_ESPERADO_ZERO:", len(_estr_0544))
    print("VALIDACAO_V23_0544NE_NEC_COMP_TOTAL_APOS_BLOQUEIO_RAIZ:", float(_estr_0544ne.get("NEC_COMP_PCS", pd.Series(dtype=float)).sum() if not _estr_0544ne.empty else 0.0))
except Exception as e:
    print("VALIDACAO_V23_0544NE_FALHOU:", e)

nec_explodida_componentes = (
    estrutura_explodida.groupby(["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO"], as_index=False)
    .agg(
        NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
        ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
        NEC_EXPLODIDA_PCS=("NEC_GERADA_LIQ_PCS", "sum"),
        MENOR_NIVEL=("NIVEL", "min")
    )
    if not estrutura_explodida.empty else pd.DataFrame(columns=["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "MENOR_NIVEL"])
)

validacao_explosao = pd.DataFrame({
    "METRICA": [
        "QTD_LINHAS_ESTRUTURA_ORIGINAL",
        "QTD_PRODUTOS_RAIZ_COM_DEMANDA",
        "QTD_LINHAS_ESTRUTURA_EXPLODIDA",
        "QTD_COMPONENTES_COM_NEC_EXPLODIDA_LIQUIDA",
        "MAIOR_NIVEL_EXPLODIDO",
        "QTD_PROBLEMAS_ESTRUTURA",
        "QTD_CICLOS_ESTRUTURA",
        "NEC_DIRETA_TOTAL_PCS",
        "NEC_EXPLODIDA_BRUTA_TOTAL_PCS",
        "ESTOQUE_CONSUMIDO_EXPLOSAO_TOTAL_PCS",
        "NEC_EXPLODIDA_LIQUIDA_TOTAL_PCS",
        "NEC_CONSOLIDADA_TOTAL_PCS",
        "QTD_ITENS_COM_DEMANDA_EXPLODIDA_LIQUIDA",
        "QTD_ITENS_EXPLODIDOS_SEM_ROTEIRO_NO_SOLVER"
    ],
    "VALOR": [
        len(estrutura_produto),
        len(demanda_raiz_estrutura),
        len(estrutura_explodida),
        nec_explodida_componentes.loc[nec_explodida_componentes["NEC_EXPLODIDA_PCS"] > TOL, "COD_INSUMO"].nunique() if not nec_explodida_componentes.empty else 0,
        estrutura_explodida["NIVEL"].max() if not estrutura_explodida.empty else 0,
        len(problemas_estrutura),
        (problemas_estrutura["MOTIVO"].eq("CICLO_ESTRUTURA").sum() if not problemas_estrutura.empty else 0),
        nec_consolidada_solver["NEC_DIRETA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["NEC_EXPLODIDA_BRUTA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["ESTOQUE_CONSUMIDO_EXPLOSAO_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["NEC_EXPLODIDA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["NEC_PCS_CONSOLIDADA"].sum() if not nec_consolidada_solver.empty else 0,
        (nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL).sum() if not nec_consolidada_solver.empty else 0,
        ((nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL) & (~nec_consolidada_solver["TEM_ROTEIRO_NO_SOLVER"])).sum() if not nec_consolidada_solver.empty else 0
    ]
})

print("Estoque componentes para explosão:", estoque_componentes.shape)
print("Estrutura explodida líquida:", estrutura_explodida.shape)
print("Necessidade explodida componentes:", nec_explodida_componentes.shape)
print("NEC consolidada solver:", nec_consolidada_solver.shape)
print("Linhas bd_mes após NEC consolidada:", bd_mes.shape)
display(validacao_explosao)


In [ ]:

# ============================================================
# 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
# ============================================================

meta_cols = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]

bd_demanda_solver = (
    bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg({**{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns}, "NEC_PCS": "max"})
)
bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

alt_cols = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "ID_RECURSO", "ID_FERRAMENTA"]
bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
bd_alternativas_item = bd_alternativas_item[bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])].copy()
bd_alternativas_item = bd_alternativas_item.drop_duplicates([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]).reset_index(drop=True)

bd_alternativas_item["ALTERNATIVA_VALIDA"] = (
    bd_alternativas_item["ALOC_REC"].notna() & bd_alternativas_item["COD_FER_UNID"].notna()
    & (bd_alternativas_item["PCS_HORA"] > TOL) & (bd_alternativas_item["HOR_REC"] > TOL)
    & (bd_alternativas_item["HOR_FER"] > TOL) & (bd_alternativas_item["HOR_CAP"] > TOL)
)
bd_alternativas_item["MOTIVO_INVALIDA"] = bd_alternativas_item.apply(classificar_motivo_invalida, axis=1)
bd_alternativas_validas = bd_alternativas_item[bd_alternativas_item["ALTERNATIVA_VALIDA"]].copy()

bd_cap_recursos = bd_alternativas_item.dropna(subset=["ALOC_REC"]).groupby("ALOC_REC", as_index=False).agg(HOR_REC=("HOR_REC", "max"))
bd_cap_ferramentas = bd_alternativas_item.dropna(subset=["COD_FER_UNID"]).groupby("COD_FER_UNID", as_index=False).agg(HOR_FER=("HOR_FER", "max"))

print("Itens com demanda:", len(bd_demanda_solver))
print("Alternativas auditadas:", len(bd_alternativas_item))
print("Alternativas válidas para produção:", len(bd_alternativas_validas))
print("Alternativas com HOR_CAP zero:", (bd_alternativas_item["HOR_CAP"] <= TOL).sum())
print("Recursos:", len(bd_cap_recursos))
print("Ferramentas:", len(bd_cap_ferramentas))


In [ ]:

# ============================================================
# 04. SOLVER - RATEIO PROPORCIONAL COM DUPLA RESTRIÇÃO
# ============================================================

def executar_rateio_proporcional_dupla_restricao(bd_demanda, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas):
    alternativas = bd_alternativas_validas.copy()
    cap_restante_rec = bd_cap_recursos.set_index("ALOC_REC")["HOR_REC"].to_dict()
    cap_restante_fer = bd_cap_ferramentas.set_index("COD_FER_UNID")["HOR_FER"].to_dict()
    pendente = bd_demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()
    producao, rastro = [], []

    niveis_prioridade = list(alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]].drop_duplicates().sort_values(["PRIOR_MATPAR", "PRIOR_ROT"]).itertuples(index=False, name=None))
    for rodada, (prior_matpar, prior_rot) in enumerate(niveis_prioridade, start=1):
        nivel = alternativas[alternativas["PRIOR_MATPAR"].eq(prior_matpar) & alternativas["PRIOR_ROT"].eq(prior_rot)].copy()
        nivel["RODADA"] = rodada
        nivel["QTD_PENDENTE_ANTES"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
        nivel = nivel[nivel["QTD_PENDENTE_ANTES"] > TOL].copy()
        if nivel.empty:
            continue

        qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
        nivel["QTD_ALTERNATIVAS_MESMO_NIVEL"] = qtd_alt_mesmo_nivel
        nivel["QTD_SOLICITADA"] = nivel["QTD_PENDENTE_ANTES"] / qtd_alt_mesmo_nivel
        nivel["HR_SOLICITADA"] = nivel["QTD_SOLICITADA"] / nivel["PCS_HORA"]

        nivel["CAP_RESTANTE_ALOC_REC_ANTES"] = nivel["ALOC_REC"].map(cap_restante_rec).fillna(0)
        nivel["CAP_RESTANTE_COD_FER_ANTES"] = nivel["COD_FER_UNID"].map(cap_restante_fer).fillna(0)

        demanda_rec = nivel.groupby("ALOC_REC")["HR_SOLICITADA"].transform("sum")
        demanda_fer = nivel.groupby("COD_FER_UNID")["HR_SOLICITADA"].transform("sum")
        nivel["HR_DEMANDA_TOTAL_ALOC_REC"] = demanda_rec
        nivel["HR_DEMANDA_TOTAL_COD_FER"] = demanda_fer
        nivel["FATOR_RATEIO_ALOC_REC"] = np.where(demanda_rec > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_ALOC_REC_ANTES"] / demanda_rec), 0.0)
        nivel["FATOR_RATEIO_COD_FER"] = np.where(demanda_fer > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_COD_FER_ANTES"] / demanda_fer), 0.0)
        nivel["FATOR_RATEIO"] = nivel[["FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER"]].min(axis=1).clip(lower=0, upper=1)
        nivel["CAP_DISPONIVEL_LINHA"] = nivel[["CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES"]].min(axis=1)

        cond_rec = nivel["FATOR_RATEIO_ALOC_REC"] < nivel["FATOR_RATEIO_COD_FER"] - TOL
        cond_fer = nivel["FATOR_RATEIO_COD_FER"] < nivel["FATOR_RATEIO_ALOC_REC"] - TOL
        cond_ambos = (~cond_rec) & (~cond_fer) & (nivel["FATOR_RATEIO"] < 1 - TOL)
        nivel["MOTIVO_LIMITANTE"] = np.select([cond_rec, cond_fer, cond_ambos], ["ALOC_REC", "COD_FER_UNID", "ALOC_REC_E_COD_FER"], default="SEM_LIMITACAO")

        nivel["HR_PRODUZIR_SOLVER"] = nivel["HR_SOLICITADA"] * nivel["FATOR_RATEIO"]
        nivel["QTD_PRODUZIR_SOLVER"] = nivel["HR_PRODUZIR_SOLVER"] * nivel["PCS_HORA"]
        nivel["QTD_PENDENTE_DEPOIS"] = (nivel["QTD_PENDENTE_ANTES"] - nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].transform("sum")).clip(lower=0)

        nivel["STATUS_DECISAO"] = np.select(
            [nivel["CAP_RESTANTE_ALOC_REC_ANTES"] <= TOL, nivel["CAP_RESTANTE_COD_FER_ANTES"] <= TOL, nivel["HR_PRODUZIR_SOLVER"] <= TOL, nivel["FATOR_RATEIO"] >= 1 - TOL, nivel["FATOR_RATEIO"] < 1 - TOL],
            ["NAO_ALOCADO_CAP_RECURSO_ZERO", "NAO_ALOCADO_CAP_FERRAMENTA_ZERO", "NAO_ALOCADO_RATEIO_ZERO", "ALOCADO_TOTAL_RODADA", "ALOCADO_PARCIAL_RATEIO"],
            default="VERIFICAR"
        )
        nivel["MOTIVO_CORTE"] = np.where(nivel["FATOR_RATEIO"] >= 1 - TOL, "SEM_CORTE", "CORTE_POR_" + nivel["MOTIVO_LIMITANTE"].astype(str))

        for aloc_rec, consumo in nivel.groupby("ALOC_REC")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_rec[aloc_rec] = max(0.0, cap_restante_rec.get(aloc_rec, 0.0) - float(consumo))
        for cod_fer, consumo in nivel.groupby("COD_FER_UNID")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_fer[cod_fer] = max(0.0, cap_restante_fer.get(cod_fer, 0.0) - float(consumo))
        for item, qtd_prod in nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum().items():
            pendente[item] = max(0.0, pendente.get(item, 0.0) - float(qtd_prod))

        producao.append(nivel[nivel["QTD_PRODUZIR_SOLVER"] > TOL].copy())
        rastro.append(nivel.copy())

    bd_producao = pd.concat(producao, ignore_index=True) if producao else pd.DataFrame()
    bd_rastro = pd.concat(rastro, ignore_index=True) if rastro else pd.DataFrame()
    return bd_producao, bd_rastro, pendente, cap_restante_rec, cap_restante_fer

bd_plano_producao, rastro_rateio, pendente_final, cap_restante_rec_final, cap_restante_fer_final = executar_rateio_proporcional_dupla_restricao(bd_demanda_solver, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas)
print("Linhas plano produção:", len(bd_plano_producao))
print("Linhas rastro rateio:", len(rastro_rateio))


In [ ]:

# ============================================================
# 05. SAÍDAS DE NEGÓCIO E RASTREABILIDADE
# ============================================================


cols_plano = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "LINHA_PROD", "FAMILIA_PROD", "MO", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "PESO_PROD_KG", "VOL_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_LIBERADA_EST"]
campos_produto_saida = ["COD_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]
campos_produto_saida = [c for c in campos_produto_saida if c in bd_mes.columns]
if campos_produto_saida and "COD_PROD" in campos_produto_saida:
    cadastro_produto_saida = bd_mes[campos_produto_saida].drop_duplicates("COD_PROD")
    for c in ["LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]:
        if c in bd_plano_producao.columns:
            bd_plano_producao = bd_plano_producao.drop(columns=[c])
    bd_plano_producao = bd_plano_producao.merge(cadastro_produto_saida, on="COD_PROD", how="left")

bd_plano_producao["PESO_PROD_KG"] = to_num(bd_plano_producao["PESO_PROD_KG"]) if "PESO_PROD_KG" in bd_plano_producao.columns else 0
bd_plano_producao["VOL_PRODUZIR_SOLVER"] = bd_plano_producao["QTD_PRODUZIR_SOLVER"] * bd_plano_producao["PESO_PROD_KG"] if "QTD_PRODUZIR_SOLVER" in bd_plano_producao.columns else 0
plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

atendido_item = plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
nao_atend_item = bd_demanda_solver.merge(atendido_item, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(lower=0)
nao_atend_item["PERC_ATENDIDO_SOLVER"] = np.where(nao_atend_item["NEC_PCS"] > 0, nao_atend_item["QTD_ATENDIDA_SOLVER"] / nao_atend_item["NEC_PCS"], 0)
nao_atend_item["PERC_NAO_ATENDIDO_SOLVER", "QTD_FINAL_VIAVEL_PAI_FILHO", "QTD_AJUSTE_PAI_FILHO", "FATOR_FINAL_VIAVEL_PAI_FILHO", "MOTIVO_AJUSTE_PAI_FILHO"] = 1 - nao_atend_item["PERC_ATENDIDO_SOLVER"]
cols_nao_atend = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER", "PERC_NAO_ATENDIDO_SOLVER"]
nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

# Ajuste pós-solver pai/filho : diagnóstico de viabilidade final corrigido e motivo textual ajustado.
ajuste_pai_filho, resumo_pai_filho, capacidade_liberada_pf = calcular_ajuste_pos_solver_pai_filho(
    estrutura_explodida,
    nec_consolidada_solver,
    nao_atend_item,
    plano_producao
)

if not resumo_pai_filho.empty:
    fatores_pf = resumo_pai_filho.rename(columns={"ID_PROD_UNID_FAT_ORIGEM": "ID_PROD_UNID_FAT"})[["MES_REF", "ID_PROD_UNID_FAT", "QTD_PAI_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "MOTIVO_AJUSTE_PAI_FILHO"]]
    nao_atend_item = nao_atend_item.merge(fatores_pf, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = nao_atend_item["QTD_PAI_FINAL_VIAVEL_EST"].fillna(nao_atend_item["QTD_ATENDIDA_SOLVER"])
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = np.minimum(nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"], nao_atend_item["QTD_ATENDIDA_SOLVER"])
    nao_atend_item["QTD_AJUSTE_PAI_FILHO"] = (nao_atend_item["QTD_ATENDIDA_SOLVER"] - nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"]).clip(lower=0)
    nao_atend_item["FATOR_FINAL_VIAVEL_PAI_FILHO"] = np.where(nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL, (nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] / nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(0, 1), 0)
    nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"] = nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"].fillna("NAO_APLICAVEL_SEM_ESTRUTURA_ORIGEM")
    nao_atend_item = nao_atend_item.drop(columns=[c for c in ["QTD_PAI_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if c in nao_atend_item.columns])

    fator_plano_pf = fatores_pf[["MES_REF", "ID_PROD_UNID_FAT", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"]]
    plano_producao = plano_producao.merge(fator_plano_pf, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"] = plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"].fillna(1.0)
    plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"] = plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"].fillna(1.0)
    plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = plano_producao["QTD_PRODUZIR_SOLVER"] * plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"]
    plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] = (plano_producao["QTD_PRODUZIR_SOLVER"] - plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"]).clip(lower=0)
    plano_producao["HR_LIBERADA_EST"] = np.where(plano_producao["PCS_HORA"] > TOL, plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] / plano_producao["PCS_HORA"], 0)
else:
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = nao_atend_item["QTD_ATENDIDA_SOLVER"]
    nao_atend_item["QTD_AJUSTE_PAI_FILHO"] = 0
    nao_atend_item["FATOR_FINAL_VIAVEL_PAI_FILHO"] = 1.0
    nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_ESTRUTURA_EXPLODIDA"
    plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"] = 1.0
    plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"] = 1.0
    plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = plano_producao["QTD_PRODUZIR_SOLVER"]
    plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] = 0
    plano_producao["HR_LIBERADA_EST"] = 0



# Correção  — blindagem contra duplicidade na saída do plano.
# A origem do problema era o merge do plano com fatores pai/filho quando o resumo pai/filho tinha mais de uma linha para a mesma chave MES_REF + ID_PROD_UNID_FAT.
# O solver/rastro não estava estourado; a duplicidade ocorria na montagem/exportação do plano e contaminava os resumos.
CHAVE_PLANO_UNICA = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','RODADA','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID','QTD_PRODUZIR_SOLVER','HR_PRODUZIR_SOLVER'] if c in plano_producao.columns]
QTD_LINHAS_PLANO_ANTES_DEDUP = len(plano_producao)
plano_producao = plano_producao.drop_duplicates(subset=CHAVE_PLANO_UNICA, keep='first').copy()
QTD_LINHAS_PLANO_DEPOIS_DEDUP = len(plano_producao)
print('Correção  - linhas plano antes/depois dedupe:', QTD_LINHAS_PLANO_ANTES_DEDUP, QTD_LINHAS_PLANO_DEPOIS_DEDUP)

prod_recurso = plano_producao.groupby("ALOC_REC", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
resumo_recursos = bd_cap_recursos.merge(prod_recurso, on="ALOC_REC", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0})
resumo_recursos["OCUPACAO_RECURSO_PCT"] = np.where(resumo_recursos["HOR_REC"] > 0, resumo_recursos["HR_PRODUZIR_SOLVER"] / resumo_recursos["HOR_REC"], 0)
resumo_recursos["HR_RECURSO_OCIOSA"] = resumo_recursos["HOR_REC"] - resumo_recursos["HR_PRODUZIR_SOLVER"]
resumo_recursos["ESTOURO_HR_RECURSO"] = np.where(resumo_recursos["HR_PRODUZIR_SOLVER"] > resumo_recursos["HOR_REC"] + TOL, resumo_recursos["HR_PRODUZIR_SOLVER"] - resumo_recursos["HOR_REC"], 0)
resumo_recursos = resumo_recursos[["ALOC_REC", "HOR_REC", "HR_PRODUZIR_SOLVER", "OCUPACAO_RECURSO_PCT", "HR_RECURSO_OCIOSA", "ESTOURO_HR_RECURSO", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"]].sort_values(["OCUPACAO_RECURSO_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

prod_ferramenta = plano_producao.groupby("COD_FER_UNID", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique"), QTD_RECURSOS=("ALOC_REC", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["COD_FER_UNID", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"])
resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on="COD_FER_UNID", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0, "QTD_RECURSOS": 0})
resumo_ferramentas["OCUPACAO_FER_PCT"] = np.where(resumo_ferramentas["HOR_FER"] > 0, resumo_ferramentas["HR_PRODUZIR_SOLVER"] / resumo_ferramentas["HOR_FER"], 0)
resumo_ferramentas["HR_FER_OCIOSA"] = resumo_ferramentas["HOR_FER"] - resumo_ferramentas["HR_PRODUZIR_SOLVER"]
resumo_ferramentas["ESTOURO_HR_FER"] = np.where(resumo_ferramentas["HR_PRODUZIR_SOLVER"] > resumo_ferramentas["HOR_FER"] + TOL, resumo_ferramentas["HR_PRODUZIR_SOLVER"] - resumo_ferramentas["HOR_FER"], 0)
resumo_ferramentas = resumo_ferramentas[["COD_FER_UNID", "HOR_FER", "HR_PRODUZIR_SOLVER", "OCUPACAO_FER_PCT", "HR_FER_OCIOSA", "ESTOURO_HR_FER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"]].sort_values(["OCUPACAO_FER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

cols_auditoria = ["MES_REF", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"]
auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].drop_duplicates().sort_values(["ALOC_REC", "COD_FER_UNID"]).copy()

cols_alt = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "ALTERNATIVA_VALIDA", "MOTIVO_INVALIDA"]
alternativas_item = bd_alternativas_item[[c for c in cols_alt if c in bd_alternativas_item.columns]].sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]).copy()

cols_rastro = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO"]
rastro_rateio = rastro_rateio[[c for c in cols_rastro if c in rastro_rateio.columns]].copy() if not rastro_rateio.empty else pd.DataFrame(columns=cols_rastro)

qtd_alt_audit = bd_alternativas_item.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_AUDITADAS=("ALOC_REC", "count"))
qtd_alt_valid = bd_alternativas_validas.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_VALIDAS=("ALOC_REC", "count"))
qtd_alt_usadas = plano_producao.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_USADAS=("ALOC_REC", "count"), QTD_ALOC_REC_USADOS=("ALOC_REC", "nunique"), QTD_COD_FER_USADOS=("COD_FER_UNID", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS"])
qtd_rodadas = rastro_rateio.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_RODADAS_TENTADAS=("RODADA", "nunique"), QTD_ALOC_REC_TENTADOS=("ALOC_REC", "nunique"), QTD_COD_FER_TENTADOS=("COD_FER_UNID", "nunique")) if not rastro_rateio.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"])

diagnostico_item = nao_atend_item.copy()
for df_merge in [qtd_alt_audit, qtd_alt_valid, qtd_alt_usadas, qtd_rodadas]:
    diagnostico_item = diagnostico_item.merge(df_merge, on="ID_PROD_UNID_FAT", how="left")
for c in ["QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"]:
    if c in diagnostico_item.columns:
        diagnostico_item[c] = diagnostico_item[c].fillna(0).astype(int)
diagnostico_item["MOTIVO_SALDO_FINAL"] = np.select([diagnostico_item["QTD_NAO_ATEND_SOLVER"] <= TOL, diagnostico_item.get("QTD_ALTERNATIVAS_AUDITADAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_VALIDAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_USADAS", 0).eq(0)], ["ATENDIDO_TOTAL", "SEM_ALTERNATIVA_CADASTRADA", "SEM_ALTERNATIVA_VALIDA", "SEM_ALOCACAO_COM_ALTERNATIVA_VALIDA"], default="SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE")

uso_alt = plano_producao.groupby(["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER"])
diagnostico_alternativas = alternativas_item.merge(uso_alt, on=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], how="left")
diagnostico_alternativas["HR_PRODUZIR_SOLVER"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"].fillna(0)
diagnostico_alternativas["QTD_PRODUZIR_SOLVER"] = diagnostico_alternativas["QTD_PRODUZIR_SOLVER"].fillna(0)
diagnostico_alternativas["ALTERNATIVA_USADA"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"] > TOL

print("Resumo recursos:", len(resumo_recursos))
print("Resumo ferramentas:", len(resumo_ferramentas))
print("Plano produção:", len(plano_producao))
print("Não atendimento:", len(nao_atend_item))
print("Rastro:", len(rastro_rateio))


In [ ]:
# ============================================================
# 06. REALOCAÇÃO DA CAPACIDADE LIBERADA - 
# ============================================================

# Nesta versão, a capacidade liberada pelo corte pai/filho volta para a máquina (ALOC_REC).
# Para realocar em outro item, o modelo valida a capacidade da máquina e também a capacidade do molde/ferramenta destino (COD_FER_UNID).
# A realocação não fica mais presa ao mesmo molde que gerou a liberação.

# ----------  realocacao ----------
def calcular_realocacao_dedup(nao_atend_item, alternativas_item, plano_producao, capacidade_liberada_pf):
    cols_log = [
        'RODADA_REALOC','MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
        'PRIOR_MATPAR','PRIOR_ROT','CLASS_ROT','ALOC_REC','COD_FER_UNID','PCS_HORA',
        'QTD_GAP_ANTES_REALOC','HR_MAQ_LIBERADA_DISP_ANTES','HR_FER_DESTINO_DISP_ANTES','HR_DISPONIVEL_REALOC',
        'QTD_POSSIVEL_REALOCAR','QTD_REALOCADA','HR_CONSUMIDA_REALOC','HR_MAQ_LIBERADA_SALDO_DEPOIS',
        'HR_FER_DESTINO_SALDO_DEPOIS','QTD_GAP_DEPOIS_REALOC','STATUS_REALOCACAO','MOTIVO_NAO_REALOCACAO','MOTIVO_AJUSTE_PAI_FILHO'
    ]
    if capacidade_liberada_pf.empty:
        return pd.DataFrame(columns=cols_log), pd.DataFrame(), pd.DataFrame()
    na = nao_atend_item.copy(); alts = alternativas_item.copy(); plano = plano_producao.copy(); cap = capacidade_liberada_pf.copy()
    for df in [na, alts, plano, cap]:
        if 'MES_REF' in df.columns:
            df['MES_REF'] = pd.to_datetime(df['MES_REF'], errors='coerce')
    for c in ['NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','QTD_ATENDIDA_SOLVER']:
        if c not in na.columns: na[c] = 0.0
        na[c] = to_num(na[c])
    na['QTD_GAP_APOS_PAI_FILHO'] = (na['NEC_PCS'] - na['QTD_FINAL_VIAVEL_PAI_FILHO']).clip(lower=0)
    # Mantem bloqueio: itens cortados por pai/filho nao recebem realocacao nesta rodada, para nao produzir sem componente.
    na['BLOQUEADO_AJUSTE_PAI_FILHO'] = na['QTD_AJUSTE_PAI_FILHO'] > TOL

    if 'ALTERNATIVA_VALIDA' in alts.columns:
        alts['_ALT_VALIDA'] = is_valid_alt(alts['ALTERNATIVA_VALIDA'])
    else:
        alts['_ALT_VALIDA'] = True
    for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT','HOR_FER','HOR_REC']:
        if c in alts.columns: alts[c] = to_num(alts[c])
    for c in ['HR_PRODUZIR_SOLVER','HOR_FER','HOR_REC']:
        if c in plano.columns: plano[c] = to_num(plano[c])
    for c in ['HR_LIBERADA_EST']:
        cap[c] = to_num(cap[c])

    # Capacidade liberada volta para a maquina, nao fica presa ao molde original.
    pool_maq = cap.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
        HR_MAQ_LIBERADA_TOTAL=('HR_LIBERADA_EST','sum'),
        QTD_ORIGENS_LIBERARAM=('ID_PROD_UNID_FAT','nunique'),
        QTD_LINHAS_LIBERARAM=('ID_PROD_UNID_FAT','count')
    )

    # Capacidade disponivel por molde destino depois do ajuste pai/filho.
    # HOR_FER total mensal por ferramenta.
    fer_cap_1 = alts.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER=('HOR_FER','max')) if 'HOR_FER' in alts.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER'])
    fer_cap_2 = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER_PLANO=('HOR_FER','max')) if 'HOR_FER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER_PLANO'])
    fer_cap = fer_cap_1.merge(fer_cap_2, on=['MES_REF','COD_FER_UNID'], how='outer')
    fer_cap['HOR_FER'] = fer_cap.get('HOR_FER', 0).fillna(0)
    if 'HOR_FER_PLANO' in fer_cap.columns:
        fer_cap['HOR_FER'] = np.maximum(fer_cap['HOR_FER'], fer_cap['HOR_FER_PLANO'].fillna(0))
    fer_cap = fer_cap[['MES_REF','COD_FER_UNID','HOR_FER']]

    fer_usado_solver = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_USADA_SOLVER=('HR_PRODUZIR_SOLVER','sum')) if 'HR_PRODUZIR_SOLVER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HR_USADA_SOLVER'])
    fer_liberada = cap.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_LIBERADA_FER=('HR_LIBERADA_EST','sum'))
    fer_pool = fer_cap.merge(fer_usado_solver, on=['MES_REF','COD_FER_UNID'], how='left').merge(fer_liberada, on=['MES_REF','COD_FER_UNID'], how='left').fillna({'HR_USADA_SOLVER':0,'HR_LIBERADA_FER':0})
    fer_pool['HR_FER_USADA_POS_PAI_FILHO'] = (fer_pool['HR_USADA_SOLVER'] - fer_pool['HR_LIBERADA_FER']).clip(lower=0)
    fer_pool['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = (fer_pool['HOR_FER'] - fer_pool['HR_FER_USADA_POS_PAI_FILHO']).clip(lower=0)

    # Candidatos: itens com gap na mesma maquina onde houve liberacao; ferramenta destino tem que ter capacidade.
    item_cols = ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','BLOQUEADO_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']
    cand = alts.merge(na[item_cols], on=['MES_REF','ID_PROD_UNID_FAT'], how='inner', suffixes=('', '_ITEM'))
    for c in ['COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
        ci = c + '_ITEM'
        if ci in cand.columns:
            cand[c] = cand[c].fillna(cand[ci]) if c in cand.columns else cand[ci]
    cand = cand[(cand['_ALT_VALIDA']) & (cand['PCS_HORA'] > TOL) & (cand['QTD_GAP_APOS_PAI_FILHO'] > TOL) & (~cand['BLOQUEADO_AJUSTE_PAI_FILHO'])].copy()
    cand = cand.merge(pool_maq[['MES_REF','ALOC_REC','HR_MAQ_LIBERADA_TOTAL']], on=['MES_REF','ALOC_REC'], how='inner')
    cand = cand.merge(fer_pool[['MES_REF','COD_FER_UNID','HR_FER_DISPONIVEL_POS_PAI_FILHO']], on=['MES_REF','COD_FER_UNID'], how='left')
    cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'].fillna(0)
    cand = cand[cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] > TOL].copy()

    gap_rem = {tuple(k): float(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['QTD_GAP_APOS_PAI_FILHO'].items()}
    bloqueado = {tuple(k): bool(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['BLOQUEADO_AJUSTE_PAI_FILHO'].items()}
    hr_maq_rem = {tuple(row[['MES_REF','ALOC_REC']]): float(row['HR_MAQ_LIBERADA_TOTAL']) for _, row in pool_maq.iterrows()}
    hr_fer_rem = {tuple(row[['MES_REF','COD_FER_UNID']]): float(row['HR_FER_DISPONIVEL_POS_PAI_FILHO']) for _, row in fer_pool.iterrows()}

    logs = []
    if not cand.empty:
        cand['_GAP_SORT'] = cand['QTD_GAP_APOS_PAI_FILHO']
        cand = cand.sort_values(['MES_REF','ALOC_REC','PRIOR_MATPAR','PRIOR_ROT','_GAP_SORT','ID_PROD_UNID_FAT'], ascending=[True, True, True, True, False, True])
        rodada = 1
        for _, row in cand.iterrows():
            item_key = (row['MES_REF'], row['ID_PROD_UNID_FAT'])
            maq_key = (row['MES_REF'], row['ALOC_REC'])
            fer_key = (row['MES_REF'], row['COD_FER_UNID'])
            gap_antes = gap_rem.get(item_key, 0.0)
            hr_maq_antes = hr_maq_rem.get(maq_key, 0.0)
            hr_fer_antes = hr_fer_rem.get(fer_key, 0.0)
            pcs_hora = float(row['PCS_HORA'])
            hr_disp = min(hr_maq_antes, hr_fer_antes)
            if gap_antes <= TOL:
                status='NAO_REALOCADO_SEM_GAP_REMANESCENTE'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            elif hr_maq_antes <= TOL:
                status='NAO_REALOCADO_SEM_HORA_MAQUINA_LIBERADA'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            elif hr_fer_antes <= TOL:
                status='NAO_REALOCADO_SEM_HORA_MOLDE_DESTINO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            elif bloqueado.get(item_key, False):
                status='NAO_REALOCADO_ITEM_BLOQUEADO_PAI_FILHO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            else:
                qtd_possivel = hr_disp * pcs_hora
                qtd_realoc = min(gap_antes, qtd_possivel)
                hr_consumida = qtd_realoc / pcs_hora if pcs_hora > TOL else 0.0
                if qtd_realoc > TOL:
                    status = 'REALOCADO_TOTAL_GAP' if qtd_realoc >= gap_antes - TOL else 'REALOCADO_PARCIAL_POR_HORA_DISPONIVEL'
                    gap_rem[item_key] = max(0.0, gap_antes - qtd_realoc)
                    hr_maq_rem[maq_key] = max(0.0, hr_maq_antes - hr_consumida)
                    hr_fer_rem[fer_key] = max(0.0, hr_fer_antes - hr_consumida)
                else:
                    status='NAO_REALOCADO_QTD_ZERO'
            logs.append({
                'RODADA_REALOC': rodada,
                'MES_REF': row['MES_REF'],
                'ID_PROD_UNID_FAT': row['ID_PROD_UNID_FAT'],
                'COD_PROD': row.get('COD_PROD'),
                'DESC_PROD': row.get('DESC_PROD'),
                'UNID_PROD': row.get('UNID_PROD'),
                'UNID_FAT': row.get('UNID_FAT'),
                'TIPO_PROD': row.get('TIPO_PROD'),
                'PRIOR_MATPAR': row.get('PRIOR_MATPAR'),
                'PRIOR_ROT': row.get('PRIOR_ROT'),
                'CLASS_ROT': row.get('CLASS_ROT'),
                'ALOC_REC': row['ALOC_REC'],
                'COD_FER_UNID': row['COD_FER_UNID'],
                'PCS_HORA': pcs_hora,
                'QTD_GAP_ANTES_REALOC': gap_antes,
                'HR_MAQ_LIBERADA_DISP_ANTES': hr_maq_antes,
                'HR_FER_DESTINO_DISP_ANTES': hr_fer_antes,
                'HR_DISPONIVEL_REALOC': hr_disp,
                'QTD_POSSIVEL_REALOCAR': qtd_possivel,
                'QTD_REALOCADA': qtd_realoc,
                'HR_CONSUMIDA_REALOC': hr_consumida,
                'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_maq_rem.get(maq_key, 0.0),
                'HR_FER_DESTINO_SALDO_DEPOIS': hr_fer_rem.get(fer_key, 0.0),
                'QTD_GAP_DEPOIS_REALOC': gap_rem.get(item_key, 0.0),
                'STATUS_REALOCACAO': status,
                'MOTIVO_NAO_REALOCACAO': '' if str(status).startswith('REALOCADO') else status,
                'MOTIVO_AJUSTE_PAI_FILHO': row.get('MOTIVO_AJUSTE_PAI_FILHO')
            })
            rodada += 1
    # : registrar sobras de horas liberadas que passaram pela realocação e não encontraram uso total.
    for (mes_ref, aloc_rec), hr_saldo in list(hr_maq_rem.items()):
        if hr_saldo > TOL:
            logs.append({
                'RODADA_REALOC': rodada if 'rodada' in locals() else 1,
                'MES_REF': mes_ref,
                'ID_PROD_UNID_FAT': '',
                'COD_PROD': '',
                'DESC_PROD': '',
                'UNID_PROD': '',
                'UNID_FAT': '',
                'TIPO_PROD': '',
                'PRIOR_MATPAR': np.nan,
                'PRIOR_ROT': np.nan,
                'CLASS_ROT': '',
                'ALOC_REC': aloc_rec,
                'COD_FER_UNID': '',
                'PCS_HORA': 0.0,
                'QTD_GAP_ANTES_REALOC': 0.0,
                'HR_MAQ_LIBERADA_DISP_ANTES': hr_saldo,
                'HR_FER_DESTINO_DISP_ANTES': 0.0,
                'HR_DISPONIVEL_REALOC': hr_saldo,
                'QTD_POSSIVEL_REALOCAR': 0.0,
                'QTD_REALOCADA': 0.0,
                'HR_CONSUMIDA_REALOC': 0.0,
                'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_saldo,
                'HR_FER_DESTINO_SALDO_DEPOIS': 0.0,
                'QTD_GAP_DEPOIS_REALOC': 0.0,
                'STATUS_REALOCACAO': 'NAO_REALOCADO_HORA_LIBERADA_REMANESCENTE',
                'MOTIVO_NAO_REALOCACAO': 'SEM_ITEM_COM_GAP_E_ALTERNATIVA_VALIDA_NA_MAQUINA_OU_SEM_FERRAMENTA_DESTINO',
                'MOTIVO_AJUSTE_PAI_FILHO': ''
            })
            rodada = (rodada + 1) if 'rodada' in locals() else 2
    realoc = pd.DataFrame(logs, columns=cols_log)
    realoc_pos = realoc[realoc['QTD_REALOCADA'] > TOL].copy() if not realoc.empty else realoc.copy()

    realoc_item = realoc_pos.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(
        QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA','sum'),
        HR_CONSUMIDA_REALOC=('HR_CONSUMIDA_REALOC','sum')
    ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_REALOCADA_CAP_LIBERADA','HR_CONSUMIDA_REALOC'])
    resumo_item = na[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']].copy()
    resumo_item = resumo_item.merge(realoc_item, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').fillna({'QTD_REALOCADA_CAP_LIBERADA':0,'HR_CONSUMIDA_REALOC':0})
    resumo_item['QTD_FINAL_APOS_REALOCACAO'] = resumo_item['QTD_FINAL_VIAVEL_PAI_FILHO'] + resumo_item['QTD_REALOCADA_CAP_LIBERADA']
    resumo_item['QTD_GAP_FINAL_APOS_REALOCACAO'] = (resumo_item['NEC_PCS'] - resumo_item['QTD_FINAL_APOS_REALOCACAO']).clip(lower=0)
    resumo_item['PERC_GAP_RECUPERADO'] = np.where(resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL, resumo_item['QTD_REALOCADA_CAP_LIBERADA'] / resumo_item['QTD_GAP_APOS_PAI_FILHO'], 0)
    resumo_item = resumo_item[resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL].sort_values(['QTD_REALOCADA_CAP_LIBERADA','QTD_GAP_APOS_PAI_FILHO'], ascending=[False, False])

    # Resumo interno para validacao (nao vai para Excel final como guia 17)
    resumo_maq = pool_maq.copy()
    uso_maq = realoc_pos.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
        HR_REAPROVEITADA=('HR_CONSUMIDA_REALOC','sum'),
        QTD_REALOCADA=('QTD_REALOCADA','sum'),
        QTD_ITENS_REALOCADOS=('ID_PROD_UNID_FAT','nunique')
    ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ALOC_REC','HR_REAPROVEITADA','QTD_REALOCADA','QTD_ITENS_REALOCADOS'])
    resumo_maq = resumo_maq.merge(uso_maq, on=['MES_REF','ALOC_REC'], how='left').fillna({'HR_REAPROVEITADA':0,'QTD_REALOCADA':0,'QTD_ITENS_REALOCADOS':0})
    resumo_maq['HR_MAQ_NAO_REAPROVEITADA'] = (resumo_maq['HR_MAQ_LIBERADA_TOTAL'] - resumo_maq['HR_REAPROVEITADA']).clip(lower=0)
    resumo_maq['PERC_REAPROVEITAMENTO_HR'] = np.where(resumo_maq['HR_MAQ_LIBERADA_TOTAL'] > TOL, resumo_maq['HR_REAPROVEITADA'] / resumo_maq['HR_MAQ_LIBERADA_TOTAL'], 0)
    return realoc, resumo_maq, resumo_item

realocacao_cap_liberada, resumo_realocacao_maquina, resumo_item_realocacao = calcular_realocacao_dedup(
    nao_atend_item,
    alternativas_item,
    plano_producao,
    capacidade_liberada_pf
)



In [ ]:

# ============================================================
# 06. VALIDAÇÕES
# ============================================================

nec_total = nao_atend_item["NEC_PCS"].sum()
atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
dif_fechamento = nec_total - atend_total - nao_atend_total
qtd_rec_estouro = (resumo_recursos["ESTOURO_HR_RECURSO"] > TOL).sum()
qtd_fer_estouro = (resumo_ferramentas["ESTOURO_HR_FER"] > TOL).sum()

validacao = pd.DataFrame({
    "METRICA": ["NEC_PCS_TOTAL", "QTD_ATENDIDA_SOLVER_TOTAL", "QTD_NAO_ATEND_SOLVER_TOTAL", "DIF_FECHAMENTO_DEMANDA", "QTD_ITENS_DEMANDA", "QTD_ITENS_COM_PRODUCAO", "QTD_ITENS_SEM_PRODUCAO", "QTD_ITENS_COM_NAO_ATENDIMENTO", "QTD_RECURSOS", "QTD_RECURSOS_ESTOURO", "ESTOURO_HR_RECURSO_TOTAL", "HOR_REC_TOTAL", "HR_PRODUZIR_SOLVER_RECURSO_TOTAL", "OCUPACAO_GLOBAL_RECURSOS", "QTD_FERRAMENTAS", "QTD_FERRAMENTAS_ESTOURO", "ESTOURO_HR_FER_TOTAL", "HOR_FER_TOTAL", "HR_PRODUZIR_SOLVER_FER_TOTAL", "OCUPACAO_GLOBAL_FERRAMENTAS", "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO", "QTD_ALTERNATIVAS_HOR_CAP_ZERO", "QTD_LINHAS_RASTRO_RATEIO", "QTD_ITENS_DIAGNOSTICO"],
    "VALOR": [nec_total, atend_total, nao_atend_total, dif_fechamento, len(nao_atend_item), (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum(), (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum(), (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum(), len(resumo_recursos), qtd_rec_estouro, resumo_recursos["ESTOURO_HR_RECURSO"].sum(), resumo_recursos["HOR_REC"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum() / resumo_recursos["HOR_REC"].sum() if resumo_recursos["HOR_REC"].sum() > 0 else 0, len(resumo_ferramentas), qtd_fer_estouro, resumo_ferramentas["ESTOURO_HR_FER"].sum(), resumo_ferramentas["HOR_FER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum() / resumo_ferramentas["HOR_FER"].sum() if resumo_ferramentas["HOR_FER"].sum() > 0 else 0, len(bd_alternativas_item), len(bd_alternativas_validas), (bd_alternativas_item["HOR_CAP"] <= TOL).sum(), len(rastro_rateio), len(diagnostico_item)]
})
validacao_checks = pd.DataFrame({"METRICA": ["FECHAMENTO_DEMANDA_OK", "CAPACIDADE_RECURSOS_OK", "CAPACIDADE_FERRAMENTAS_OK"], "VALOR": [abs(dif_fechamento) <= 1e-4, qtd_rec_estouro == 0, qtd_fer_estouro == 0]})
validacao = pd.concat([validacao, pd.DataFrame({"METRICA": ["---"], "VALOR": [""]}), validacao_checks], ignore_index=True)
display(validacao)


# Adiciona bloco de validação da explosão de estrutura dentro da validação geral
validacao = pd.concat([
    validacao,
    pd.DataFrame({"METRICA": ["--- EXPLOSAO_ESTRUTURA ---"], "VALOR": [""]}),
    validacao_explosao
], ignore_index=True)


# Validação específica  - componente 0544NE recebendo demanda estrutural do pai 0544
try:
    validacao_0544ne = bd_mes.loc[bd_mes["COD_PROD"].astype(str).eq("0544NE"), [c for c in ["MES_REF","ID_PROD_UNID_FAT","COD_PROD","UNID_PROD","UNID_FAT","NEC_DIRETA_PCS","NEC_COMP_PCS","NEC_PCS"] if c in bd_mes.columns]].drop_duplicates().copy()
    print("Validação  - linhas 0544NE com NEC_COMP_PCS > 0:", int((validacao_0544ne.get("NEC_COMP_PCS", 0) > TOL).sum()))
    display(validacao_0544ne)
except Exception as e:
    print("Aviso validação 0544NE não executada:", e)


In [ ]:

# ============================================================
# 06B. AUDITORIA COMPLETA DO PLANO 
# ============================================================
# AUDITORIA_PLANO_COMPLETO_STATUS_ZERO_AUDIT
# Regra: toda combinação MES_REF + ID_PROD_UNID_FAT existente no bd_LTP_NEC_SOLVER
# deve aparecer no 03_PLANO_PRODUCAO.
# Se QTD_PRODUZIR_SOLVER = 0, MOTIVO_STATUS deve explicar o motivo correto:
# - DEMANDA_DIRETA_COBERTA_POR_ESTOQUE
# - DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE
# - ITEM_SEM_DEMANDA_NO_MES
# - NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE
# - NAO_ALOCADO_RECURSO_SEM_CAPACIDADE
# - NAO_ALOCADO_PCS_HORA_INVALIDO
# - NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA
# - FALHA_RETORNO_DEMANDA_COMPONENTE_LTP

try:
    def _key_mes_id(df):
        return pd.to_datetime(df["MES_REF"], errors="coerce").dt.strftime("%Y-%m-%d") + "|" + df["ID_PROD_UNID_FAT"].astype(str)

    def _calc_demanda_bruta_direta_ref(df):
        cols = ["LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS"]
        bruta = sum([to_num(df[c]) if c in df.columns else 0 for c in cols])
        if "MESMA_REG" in df.columns and "LTP_SALDO_PREV_PROX_MES_PCS" in df.columns:
            bruta = bruta + np.where(df["MESMA_REG"].astype(str).str.upper().eq("NAO"), to_num(df["LTP_SALDO_PREV_PROX_MES_PCS"]), 0)
        return pd.Series(bruta, index=df.index)

    def _calc_cobertura_ref(df):
        cols = ["LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS"]
        return sum([to_num(df[c]) if c in df.columns else 0 for c in cols])

    def _map_status_decisao_audit(motivo):
        if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE"]:
            return "NAO_PLANEJADO_DEMANDA_COBERTA_ESTOQUE"
        if motivo == "ITEM_SEM_DEMANDA_NO_MES":
            return "NAO_PLANEJADO_SEM_DEMANDA"
        if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
            return "NAO_ALOCADO_CAP_FERRAMENTA_ZERO"
        if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
            return "NAO_ALOCADO_CAP_RECURSO_ZERO"
        if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
            return "NAO_ALOCADO_ALTERNATIVA_INVALIDA"
        if motivo == "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA":
            return "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA"
        if motivo == "FALHA_RETORNO_DEMANDA_COMPONENTE_LTP":
            return "NAO_PLANEJADO_ALERTA_COMPONENTE"
        return "NAO_PLANEJADO_SEM_DEMANDA"

    def _motivo_limitante_audit(motivo):
        if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
            return "COD_FER_UNID"
        if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
            return "ALOC_REC"
        if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
            return "PCS_HORA_INVALIDO"
        if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"]:
            return "NAO_APLICAVEL_SEM_DEMANDA_LIQUIDA"
        return "NAO_APLICAVEL"

    def _motivo_corte_audit(motivo):
        if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
            return "CORTE_POR_COD_FER_UNID"
        if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
            return "CORTE_POR_ALOC_REC"
        if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
            return "CORTE_POR_ALTERNATIVA_INVALIDA"
        if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"]:
            return "SEM_CORTE_SEM_DEMANDA_LIQUIDA"
        return "SEM_CORTE"

    base_ltp_audit = bd_mes.copy()
    base_ltp_audit["MES_REF"] = pd.to_datetime(base_ltp_audit["MES_REF"], errors="coerce")
    base_ltp_audit["DEMANDA_BRUTA_DIRETA_REF_PCS"] = _calc_demanda_bruta_direta_ref(base_ltp_audit)
    base_ltp_audit["COBERTURA_ESTOQUE_REF_PCS"] = _calc_cobertura_ref(base_ltp_audit)
    base_ltp_audit["NEC_DIRETA_CALC_REF_PCS"] = (base_ltp_audit["DEMANDA_BRUTA_DIRETA_REF_PCS"] - base_ltp_audit["COBERTURA_ESTOQUE_REF_PCS"]).clip(lower=0)

    base_cols = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "DEMANDA_BRUTA_DIRETA_REF_PCS", "COBERTURA_ESTOQUE_REF_PCS", "NEC_DIRETA_CALC_REF_PCS"] if c in base_ltp_audit.columns]
    base_status = base_ltp_audit.sort_values([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in base_ltp_audit.columns])[base_cols].drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"], keep="first").copy()

    nec_cols = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"] if c in nec_consolidada_solver.columns]
    if nec_cols:
        base_status = base_status.merge(nec_consolidada_solver[nec_cols].drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"]), on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")

    # Motivo de alternativa inválida para itens com necessidade líquida, mas sem produção.
    motivos_alt = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "MOTIVOS_INVALIDA"])
    if "MOTIVO_INVALIDA" in bd_alternativas_item.columns:
        motivos_alt = bd_alternativas_item.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(MOTIVOS_INVALIDA=("MOTIVO_INVALIDA", lambda x: " | ".join(sorted(set(map(str, x))))))
        base_status = base_status.merge(motivos_alt, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    else:
        base_status["MOTIVOS_INVALIDA"] = ""

    def _classificar_status_audit(row):
        nec_total = float(row.get("NEC_PCS_CONSOLIDADA", 0) or 0)
        direta = float(row.get("NEC_DIRETA_PCS", 0) or 0)
        comp = float(row.get("NEC_COMP_PCS", 0) or 0)
        exp_bruta = float(row.get("NEC_EXPLODIDA_BRUTA_PCS", 0) or 0)
        exp_liq = float(row.get("NEC_EXPLODIDA_PCS", 0) or 0)
        bruta_dir = float(row.get("DEMANDA_BRUTA_DIRETA_REF_PCS", 0) or 0)
        cobertura = float(row.get("COBERTURA_ESTOQUE_REF_PCS", 0) or 0)
        motivos = str(row.get("MOTIVOS_INVALIDA", ""))
        if nec_total <= TOL:
            if bruta_dir > TOL and cobertura + TOL >= bruta_dir:
                return "DEMANDA_DIRETA_COBERTA_POR_ESTOQUE"
            if exp_bruta > TOL and exp_liq <= TOL:
                return "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE"
            if exp_bruta > TOL and exp_liq > TOL and comp <= TOL:
                return "FALHA_RETORNO_DEMANDA_COMPONENTE_LTP"
            return "ITEM_SEM_DEMANDA_NO_MES"
        if "HOR_FER_ZERO_OU_NEGATIVO" in motivos:
            return "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE"
        if "HOR_REC_ZERO_OU_NEGATIVO" in motivos:
            return "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE"
        if "PCS_HORA_ZERO_OU_INVALIDO" in motivos:
            return "NAO_ALOCADO_PCS_HORA_INVALIDO"
        return "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA"

    base_status["STATUS_SUGERIDO_AUDIT"] = base_status.apply(_classificar_status_audit, axis=1)
    base_status["_KEY_AUDIT"] = _key_mes_id(base_status)
    plano_producao["_KEY_AUDIT"] = _key_mes_id(plano_producao)

    diag_cols_audit = ["DEMANDA_BRUTA_DIRETA_REF_PCS", "COBERTURA_ESTOQUE_REF_PCS", "NEC_DIRETA_CALC_REF_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"]
    for c in diag_cols_audit:
        if c not in plano_producao.columns:
            plano_producao[c] = np.nan

    base_merge_cols = ["_KEY_AUDIT", "STATUS_SUGERIDO_AUDIT"] + [c for c in diag_cols_audit if c in base_status.columns]
    plano_producao = plano_producao.merge(base_status[base_merge_cols].drop_duplicates("_KEY_AUDIT"), on="_KEY_AUDIT", how="left", suffixes=("", "_AUD"))
    for c in diag_cols_audit:
        ca = c + "_AUD"
        if ca in plano_producao.columns:
            plano_producao[c] = plano_producao[c].where(plano_producao[c].notna(), plano_producao[ca])
            plano_producao = plano_producao.drop(columns=[ca])

    if "MOTIVO_STATUS" not in plano_producao.columns:
        plano_producao["MOTIVO_STATUS"] = ""
    qtd_plan = to_num(plano_producao["QTD_PRODUZIR_SOLVER"])
    plano_producao.loc[qtd_plan > TOL, "MOTIVO_STATUS"] = "PRODUCAO_ALOCADA_PELO_SOLVER"
    mask_zero = qtd_plan <= TOL
    plano_producao.loc[mask_zero & plano_producao["STATUS_SUGERIDO_AUDIT"].notna(), "MOTIVO_STATUS"] = plano_producao.loc[mask_zero & plano_producao["STATUS_SUGERIDO_AUDIT"].notna(), "STATUS_SUGERIDO_AUDIT"]
    for idx in plano_producao.index[mask_zero]:
        motivo = str(plano_producao.at[idx, "MOTIVO_STATUS"])
        plano_producao.at[idx, "STATUS_DECISAO"] = _map_status_decisao_audit(motivo)
        if "MOTIVO_LIMITANTE" in plano_producao.columns:
            plano_producao.at[idx, "MOTIVO_LIMITANTE"] = _motivo_limitante_audit(motivo)
        if "MOTIVO_CORTE" in plano_producao.columns:
            plano_producao.at[idx, "MOTIVO_CORTE"] = _motivo_corte_audit(motivo)

    keys_plano = set(plano_producao["_KEY_AUDIT"].astype(str))
    faltantes_audit = base_status[~base_status["_KEY_AUDIT"].astype(str).isin(keys_plano)].copy()
    linhas_zero = []
    for _, r in faltantes_audit.iterrows():
        row = {c: 0 for c in plano_producao.columns if c != "STATUS_SUGERIDO_AUDIT"}
        for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP"] + diag_cols_audit:
            if c in row and c in r.index:
                row[c] = r[c]
        motivo = str(r.get("STATUS_SUGERIDO_AUDIT", "ITEM_SEM_DEMANDA_NO_MES"))
        row["STATUS_DECISAO"] = _map_status_decisao_audit(motivo)
        row["MOTIVO_STATUS"] = motivo
        if "MOTIVO_LIMITANTE" in row:
            row["MOTIVO_LIMITANTE"] = _motivo_limitante_audit(motivo)
        if "MOTIVO_CORTE" in row:
            row["MOTIVO_CORTE"] = _motivo_corte_audit(motivo)
        if "RODADA" in row:
            row["RODADA"] = 0
        row["_KEY_AUDIT"] = r["_KEY_AUDIT"]
        linhas_zero.append(row)

    if linhas_zero:
        plano_producao = pd.concat([plano_producao, pd.DataFrame(linhas_zero).reindex(columns=plano_producao.columns)], ignore_index=True)

    plano_producao = plano_producao.drop(columns=[c for c in ["STATUS_SUGERIDO_AUDIT", "_KEY_AUDIT"] if c in plano_producao.columns])
    print("Auditoria  - combinações LTP incluídas no plano:", len(linhas_zero))
    print(plano_producao["MOTIVO_STATUS"].value_counts(dropna=False))
except Exception as e:
    print("Aviso auditoria  plano completo:", e)


In [ ]:

# ============================================================
# 06C. CORREÇÃO  - PLANO FINAL SEM ALTERAR LAYOUT
# ============================================================
# Regra :
# - não criar coluna nova para o cliente;
# - não renomear coluna;
# - não remover coluna;
# - manter QTD_PRODUZIR_SOLVER como coluna oficial da 03_PLANO_PRODUCAO;
# - atualizar apenas o valor de QTD_PRODUZIR_SOLVER para a produção final válida,
#   usando as colunas já existentes desde a .

print('Aplicando correção : QTD_PRODUZIR_SOLVER final na 03_PLANO_PRODUCAO, sem alterar layout.')

_keys_plano = ['MES_REF', 'ID_PROD_UNID_FAT']
if 'MES_REF' in plano_producao.columns:
    plano_producao['MES_REF'] = pd.to_datetime(plano_producao['MES_REF'], errors='coerce')
if 'MES_REF' in nao_atend_item.columns:
    nao_atend_item['MES_REF'] = pd.to_datetime(nao_atend_item['MES_REF'], errors='coerce')

# Restaura colunas já validadas na  quando a execução local ainda não as tiver montado.
for _c in ['MOTIVO_AJUSTE_PAI_FILHO', 'QTD_REALOCADA_CAP_LIBERADA', 'QTD_FINAL_APOS_REALOCACAO', 'QTD_GAP_FINAL_APOS_REALOCACAO']:
    if _c not in plano_producao.columns:
        plano_producao[_c] = '' if _c == 'MOTIVO_AJUSTE_PAI_FILHO' else 0.0

if 'MOTIVO_AJUSTE_PAI_FILHO' in nao_atend_item.columns:
    _mot_pf_plano = nao_atend_item[_keys_plano + ['MOTIVO_AJUSTE_PAI_FILHO']].drop_duplicates(_keys_plano)
    plano_producao = plano_producao.merge(_mot_pf_plano, on=_keys_plano, how='left', suffixes=('', '_NA_PLANO'))
    if 'MOTIVO_AJUSTE_PAI_FILHO_NA_PLANO' in plano_producao.columns:
        plano_producao['MOTIVO_AJUSTE_PAI_FILHO'] = np.where(
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO'].astype(str).str.len() > 0,
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO'],
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO_NA_PLANO'].fillna('')
        )
        plano_producao = plano_producao.drop(columns=['MOTIVO_AJUSTE_PAI_FILHO_NA_PLANO'])

# Restaura colunas da guia 04 que já existiam na .
if 'FATOR_AJUSTE_SOBRE_SOLVER' not in nao_atend_item.columns:
    nao_atend_item['FATOR_AJUSTE_SOBRE_SOLVER'] = nao_atend_item['FATOR_FINAL_VIAVEL_PAI_FILHO'] if 'FATOR_FINAL_VIAVEL_PAI_FILHO' in nao_atend_item.columns else 1.0
if 'QTD_GAP_APOS_PAI_FILHO' not in nao_atend_item.columns:
    _nec_na_plano = to_num(nao_atend_item['NEC_PCS']) if 'NEC_PCS' in nao_atend_item.columns else 0.0
    _final_na_plano = to_num(nao_atend_item['QTD_FINAL_VIAVEL_PAI_FILHO']) if 'QTD_FINAL_VIAVEL_PAI_FILHO' in nao_atend_item.columns else to_num(nao_atend_item['QTD_ATENDIDA_SOLVER'])
    nao_atend_item['QTD_GAP_APOS_PAI_FILHO'] = np.maximum(_nec_na_plano - _final_na_plano, 0)
if 'BLOQUEADO_AJUSTE_PAI_FILHO' not in nao_atend_item.columns:
    _mot_na_plano = nao_atend_item['MOTIVO_AJUSTE_PAI_FILHO'].astype(str) if 'MOTIVO_AJUSTE_PAI_FILHO' in nao_atend_item.columns else ''
    nao_atend_item['BLOQUEADO_AJUSTE_PAI_FILHO'] = _mot_na_plano.eq('FILHO_COM_ROTEIRO_LIMITOU_PAI')

# Base de realocação por linha produtiva.
# : a quantidade realocada deve ser somada na alternativa em que a realocação ocorreu.
# Não distribuir a realocação proporcionalmente por todas as linhas do item, para não criar estouro artificial de capacidade.
_realoc_linha_plano = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT','QTD_REALOCADA_CAP_LIBERADA'])
try:
    if 'realocacao_cap_liberada' in globals() and isinstance(realocacao_cap_liberada, pd.DataFrame) and not realocacao_cap_liberada.empty:
        _keys_realoc_linha_plano = ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
        _cols_realoc_linha_plano = [c for c in _keys_realoc_linha_plano + ['QTD_REALOCADA'] if c in realocacao_cap_liberada.columns]
        if set(_keys_realoc_linha_plano).issubset(_cols_realoc_linha_plano) and 'QTD_REALOCADA' in _cols_realoc_linha_plano:
            _realoc_linha_plano = realocacao_cap_liberada[_cols_realoc_linha_plano].copy()
            _realoc_linha_plano['MES_REF'] = pd.to_datetime(_realoc_linha_plano['MES_REF'], errors='coerce')
            _realoc_linha_plano['QTD_REALOCADA'] = to_num(_realoc_linha_plano['QTD_REALOCADA'])
            _realoc_linha_plano = _realoc_linha_plano[_realoc_linha_plano['QTD_REALOCADA'] > TOL].copy()
            _realoc_linha_plano = _realoc_linha_plano.groupby(_keys_realoc_linha_plano, as_index=False).agg(QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA', 'sum'))
except Exception as e:
    print('Aviso : não foi possível usar realocacao_cap_liberada por linha:', e)

plano_producao['_QTD_BASE_FINAL_PLANO'] = to_num(plano_producao['QTD_PRODUZIR_FINAL_VIAVEL_EST']) if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in plano_producao.columns else to_num(plano_producao['QTD_PRODUZIR_SOLVER'])
if not _realoc_linha_plano.empty:
    _keys_merge_realoc_plano = ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
    plano_producao = plano_producao.merge(_realoc_linha_plano, on=_keys_merge_realoc_plano, how='left', suffixes=('', '_LINHA_PLANO'))
    if 'QTD_REALOCADA_CAP_LIBERADA_LINHA_PLANO' in plano_producao.columns:
        plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = to_num(plano_producao['QTD_REALOCADA_CAP_LIBERADA_LINHA_PLANO'])
        plano_producao = plano_producao.drop(columns=['QTD_REALOCADA_CAP_LIBERADA_LINHA_PLANO'])
    else:
        plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0
else:
    plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0

plano_producao['QTD_FINAL_APOS_REALOCACAO'] = plano_producao['_QTD_BASE_FINAL_PLANO'] + to_num(plano_producao['QTD_REALOCADA_CAP_LIBERADA'])

if 'NEC_PCS_CONSOLIDADA' in plano_producao.columns:
    _nec_ref_plano = to_num(plano_producao['NEC_PCS_CONSOLIDADA'])
elif 'NEC_PCS' in plano_producao.columns:
    _nec_ref_plano = to_num(plano_producao['NEC_PCS'])
else:
    _nec_ref_plano = 0.0
plano_producao['QTD_GAP_FINAL_APOS_REALOCACAO'] = np.maximum(_nec_ref_plano - to_num(plano_producao['QTD_FINAL_APOS_REALOCACAO']), 0)

# Atualiza a coluna oficial sem mudar o nome.
plano_producao['QTD_PRODUZIR_SOLVER'] = to_num(plano_producao['QTD_FINAL_APOS_REALOCACAO'])
plano_producao['HR_PRODUZIR_SOLVER'] = np.where(to_num(plano_producao['PCS_HORA']) > TOL, to_num(plano_producao['QTD_PRODUZIR_SOLVER']) / to_num(plano_producao['PCS_HORA']), 0)
if 'PESO_PROD_KG' in plano_producao.columns and 'VOL_PRODUZIR_SOLVER' in plano_producao.columns:
    plano_producao['VOL_PRODUZIR_SOLVER'] = to_num(plano_producao['QTD_PRODUZIR_SOLVER']) * to_num(plano_producao['PESO_PROD_KG'])
plano_producao = plano_producao.drop(columns=['_QTD_BASE_FINAL_PLANO'])

# Recalcula resumos que dependem diretamente da coluna oficial, sem mudar layout.
prod_recurso = plano_producao.groupby('ALOC_REC', as_index=False).agg(
    HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
    QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
    QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique')
) if not plano_producao.empty else pd.DataFrame(columns=['ALOC_REC', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS'])
resumo_recursos = bd_cap_recursos.merge(prod_recurso, on='ALOC_REC', how='left').fillna({'HR_PRODUZIR_SOLVER': 0, 'QTD_PRODUZIR_SOLVER': 0, 'QTD_ITENS': 0})
resumo_recursos['OCUPACAO_RECURSO_PCT'] = np.where(resumo_recursos['HOR_REC'] > 0, resumo_recursos['HR_PRODUZIR_SOLVER'] / resumo_recursos['HOR_REC'], 0)
resumo_recursos['HR_RECURSO_OCIOSA'] = resumo_recursos['HOR_REC'] - resumo_recursos['HR_PRODUZIR_SOLVER']
resumo_recursos['ESTOURO_HR_RECURSO'] = np.where(resumo_recursos['HR_PRODUZIR_SOLVER'] > resumo_recursos['HOR_REC'] + TOL, resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC'], 0)
resumo_recursos = resumo_recursos[['ALOC_REC', 'HOR_REC', 'HR_PRODUZIR_SOLVER', 'OCUPACAO_RECURSO_PCT', 'HR_RECURSO_OCIOSA', 'ESTOURO_HR_RECURSO', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS']].sort_values(['OCUPACAO_RECURSO_PCT', 'HR_PRODUZIR_SOLVER'], ascending=[False, False])

prod_ferramenta = plano_producao.groupby('COD_FER_UNID', as_index=False).agg(
    HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
    QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
    QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique'),
    QTD_RECURSOS=('ALOC_REC', 'nunique')
) if not plano_producao.empty else pd.DataFrame(columns=['COD_FER_UNID', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS', 'QTD_RECURSOS'])
resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on='COD_FER_UNID', how='left').fillna({'HR_PRODUZIR_SOLVER': 0, 'QTD_PRODUZIR_SOLVER': 0, 'QTD_ITENS': 0, 'QTD_RECURSOS': 0})
resumo_ferramentas['OCUPACAO_FER_PCT'] = np.where(resumo_ferramentas['HOR_FER'] > 0, resumo_ferramentas['HR_PRODUZIR_SOLVER'] / resumo_ferramentas['HOR_FER'], 0)
resumo_ferramentas['HR_FER_OCIOSA'] = resumo_ferramentas['HOR_FER'] - resumo_ferramentas['HR_PRODUZIR_SOLVER']
resumo_ferramentas['ESTOURO_HR_FER'] = np.where(resumo_ferramentas['HR_PRODUZIR_SOLVER'] > resumo_ferramentas['HOR_FER'] + TOL, resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER'], 0)
resumo_ferramentas = resumo_ferramentas[['COD_FER_UNID', 'HOR_FER', 'HR_PRODUZIR_SOLVER', 'OCUPACAO_FER_PCT', 'HR_FER_OCIOSA', 'ESTOURO_HR_FER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS', 'QTD_RECURSOS']].sort_values(['OCUPACAO_FER_PCT', 'HR_PRODUZIR_SOLVER'], ascending=[False, False])

# Recalcula diagnóstico de alternativas para manter consistência com a coluna oficial.
try:
    uso_alt = plano_producao.groupby(['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT'], as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum')
    ) if not plano_producao.empty else pd.DataFrame(columns=['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER'])
    diagnostico_alternativas = alternativas_item.merge(uso_alt, on=['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT'], how='left')
    diagnostico_alternativas['HR_PRODUZIR_SOLVER'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'].fillna(0)
    diagnostico_alternativas['QTD_PRODUZIR_SOLVER'] = diagnostico_alternativas['QTD_PRODUZIR_SOLVER'].fillna(0)
    diagnostico_alternativas['ALTERNATIVA_USADA'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'] > TOL
except Exception as e:
    print('Aviso : diagnóstico de alternativas não recalculado:', e)


# Preserva exatamente a ordem de colunas já validada na  para as guias afetadas.
_ORDEM_03_PLANO_PLANO = ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'LINHA_PROD', 'FAMILIA_PROD', 'MO', 'RODADA', 'PRIOR_MATPAR', 'PRIOR_ROT', 'CLASS_ROT', 'ALOC_REC', 'COD_FER_UNID', 'PCS_HORA', 'HOR_REC', 'HOR_FER', 'HOR_CAP', 'QTD_PENDENTE_ANTES', 'QTD_SOLICITADA', 'HR_SOLICITADA', 'CAP_RESTANTE_ALOC_REC_ANTES', 'CAP_RESTANTE_COD_FER_ANTES', 'CAP_DISPONIVEL_LINHA', 'HR_DEMANDA_TOTAL_ALOC_REC', 'HR_DEMANDA_TOTAL_COD_FER', 'FATOR_RATEIO_ALOC_REC', 'FATOR_RATEIO_COD_FER', 'FATOR_RATEIO', 'MOTIVO_LIMITANTE', 'QTD_PRODUZIR_SOLVER', 'PESO_PROD_KG', 'VOL_PRODUZIR_SOLVER', 'HR_PRODUZIR_SOLVER', 'QTD_PENDENTE_DEPOIS', 'MOTIVO_CORTE', 'STATUS_DECISAO', 'DEMANDA_BRUTA_DIRETA_REF_PCS', 'COBERTURA_ESTOQUE_REF_PCS', 'NEC_DIRETA_CALC_REF_PCS', 'NEC_DIRETA_PCS', 'NEC_COMP_PCS', 'NEC_EXPLODIDA_BRUTA_PCS', 'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS', 'NEC_PCS_CONSOLIDADA', 'MOTIVO_STATUS', 'QTD_REDUCAO_POS_PAI_FILHO', 'FATOR_FINAL_VIAVEL_ORIGEM', 'FATOR_AJUSTE_SOBRE_SOLVER', 'MOTIVO_AJUSTE_PAI_FILHO', 'QTD_PRODUZIR_FINAL_VIAVEL_EST', 'HR_LIBERADA_EST', 'QTD_REALOCADA_CAP_LIBERADA', 'QTD_FINAL_APOS_REALOCACAO', 'QTD_GAP_FINAL_APOS_REALOCACAO']
_ORDEM_04_NAO_ATEND_PLANO = ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'NEC_PCS_ORIGINAL', 'NEC_ESTRUTURA_PAI_PCS', 'NEC_DIRETA_BRUTA_PCS', 'NEC_DIRETA_PCS', 'NEC_EXPLODIDA_BRUTA_PCS', 'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS', 'NEC_COMP_PCS', 'NEC_PCS_CONSOLIDADA', 'NEC_PCS', 'QTD_ATENDIDA_SOLVER', 'QTD_NAO_ATEND_SOLVER', 'PERC_ATENDIDO_SOLVER', 'QTD_FINAL_VIAVEL_PAI_FILHO', 'QTD_AJUSTE_PAI_FILHO', 'FATOR_FINAL_VIAVEL_PAI_FILHO', 'FATOR_AJUSTE_SOBRE_SOLVER', 'MOTIVO_AJUSTE_PAI_FILHO', 'QTD_GAP_APOS_PAI_FILHO', 'BLOQUEADO_AJUSTE_PAI_FILHO']
plano_producao = plano_producao[[c for c in _ORDEM_03_PLANO_PLANO if c in plano_producao.columns] + [c for c in plano_producao.columns if c not in _ORDEM_03_PLANO_PLANO]].copy()
nao_atend_item = nao_atend_item[[c for c in _ORDEM_04_NAO_ATEND_PLANO if c in nao_atend_item.columns] + [c for c in nao_atend_item.columns if c not in _ORDEM_04_NAO_ATEND_PLANO]].copy()

print('Correção  aplicada sem criar, renomear ou remover colunas além das já existentes na .')


In [ ]:

# ============================================================
# 06D.  - CORRECAO DO CORTE POR COMPONENTE NO FECHAMENTO DO M1
# ============================================================
# Regras consolidadas :
# - ESTOQUE reduz a necessidade liquida do componente; estoque nao corta pai diretamente;
# - pai so e reduzido quando sobra corte real do componente apos a otimizacao de capacidade;
# - o corte do componente e rateado proporcionalmente entre demanda direta e origens estruturais;
# - monta uma vez o grafo estrutural a partir da explosão validada;
# - as quantidades passam a nascer da QTD_PRODUZIR_SOLVER final atual do pai;
# - pai mudou -> recalcula filhos;
# - filho mudou -> recalcula pais que dependem dele;
# - capacidade liberada por cortes volta para tentativa de realocação na mesma unidade/máquina,
#   validando a ferramenta/molde destino;
# - a realocação também volta para o ciclo estrutural;
# - não chama a explosão completa nem consolidação pesada dentro do loop;
# - não cria, remove ou renomeia colunas de saída.

from collections import defaultdict, deque

MAX_ITER_FECHAMENTO_M1 = 60
TOL_FECHAMENTO_M1 = 1e-9
APLICAR_REALOCACAO_M1 = True  #  prioriza fechamento pai/filho; realocacao fica bloqueada se puder quebrar fechamento/capacidade.


def _num_m1(s):
    return pd.to_numeric(s, errors='coerce').fillna(0.0)


def _normalizar_m1(pl):
    pl = pl.copy()
    if 'MES_REF' in pl.columns:
        pl['MES_REF'] = pd.to_datetime(pl['MES_REF'], errors='coerce')
    for c in [
        'QTD_PRODUZIR_SOLVER','HR_PRODUZIR_SOLVER','PCS_HORA','NEC_DIRETA_PCS',
        'NEC_COMP_PCS','NEC_EXPLODIDA_BRUTA_PCS','ESTOQUE_CONSUMIDO_EXPLOSAO_PCS',
        'NEC_EXPLODIDA_PCS','NEC_PCS_CONSOLIDADA','QTD_REDUCAO_POS_PAI_FILHO',
        'QTD_PRODUZIR_FINAL_VIAVEL_EST','HR_LIBERADA_EST','QTD_REALOCADA_CAP_LIBERADA',
        'QTD_FINAL_APOS_REALOCACAO','QTD_GAP_FINAL_APOS_REALOCACAO'
    ]:
        if c in pl.columns:
            pl[c] = _num_m1(pl[c])
    return pl


def _prod_item_m1(pl):
    if pl.empty:
        return pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_ITEM'])
    return (
        pl.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False)
        .agg(QTD_ITEM=('QTD_PRODUZIR_SOLVER','sum'))
    )


def _meta_item_m1():
    # Uma linha por item/chave com estoque fisico autoritativo e regras operacionais.
    base = bd_mes.copy() if 'bd_mes' in globals() else nec_consolidada_solver.copy()
    base['MES_REF'] = pd.to_datetime(base['MES_REF'], errors='coerce')
    for c in ['ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
        if c in base.columns:
            base[c] = base[c].astype(str)
    num_cols = ['NEC_DIRETA_PCS','NEC_DIRETA_BRUTA_PCS','NEC_PCS_CONSOLIDADA','NEC_PCS',
                'LTP_EST_INI_PCS','LTP_EST_TRANS_PCS','ORI_TOT_PCS','TRIANG_TOT_PCS',
                'LIMIT_PCS','LOTE_MIN','QTD_EMB']
    for c in num_cols:
        if c not in base.columns: base[c] = 0.0
        base[c] = _num_m1(base[c])
    base['ESTOQUE_FISICO_ORIGEM'] = (base['LTP_EST_INI_PCS'] + base['LTP_EST_TRANS_PCS'] + base['ORI_TOT_PCS'] + base['TRIANG_TOT_PCS']).clip(lower=0.0)
    if 'NEC_DIRETA_BRUTA_PCS' not in base.columns:
        base['NEC_DIRETA_BRUTA_PCS'] = base['NEC_DIRETA_PCS']
    agg = {c:'first' for c in ['COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_DIRETA_PCS','NEC_DIRETA_BRUTA_PCS','NEC_PCS_CONSOLIDADA','NEC_PCS','ESTOQUE_FISICO_ORIGEM','LIMIT_PCS','LOTE_MIN','QTD_EMB'] if c in base.columns}
    meta = base.sort_values([c for c in ['MES_REF','ID_PROD_UNID_FAT','PRIOR_MATPAR','PRIOR_ROT'] if c in base.columns]).groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False).agg(agg)
    return meta


def _flags_operacionais_m1():
    # Regra oficial: se flag nao existir/nao puder ser lida, assumir SIM.
    lote = globals().get('LOTE_MIN_FLAG', True)
    emb = globals().get('MULTIPLO_EMB_FLAG', True)
    def ativo(v):
        if v is None: return True
        if isinstance(v,bool): return v
        t=str(v).strip().upper()
        if not t: return True
        return t not in {'NAO','NÃO','FALSE','0','DESATIVADO'}
    return ativo(lote), ativo(emb)


def _quantidade_operacional_alvo_m1(qtd_liquida, row):
    # Normaliza PARA CIMA apenas quando existe necessidade liquida positiva.
    q=float(max(0.0,qtd_liquida))
    if q <= TOL_FECHAMENTO_M1: return 0.0
    q=float(np.ceil(q))  # pecas inteiras
    lim=float(max(0.0,row.get('LIMIT_PCS',0) or 0))
    lote=float(max(0.0,row.get('LOTE_MIN',0) or 0))
    emb=float(max(0.0,row.get('QTD_EMB',0) or 0))
    tipo=str(row.get('TIPO_PROD','')).strip().upper()
    flag_lote, flag_emb = _flags_operacionais_m1()
    q=max(q, float(np.ceil(lim)))
    if flag_lote and lote>0: q=max(q,float(np.ceil(lote)))
    if flag_emb and tipo in {'PA','MR'} and emb>0:
        q=float(np.ceil(q/emb)*emb)
    return q


def _quantidade_valida_abaixo_m1(limite, row):
    # Maior quantidade operacional valida <= limite. Se nao comportar primeiro lote valido, retorna zero.
    cap=float(max(0.0,limite))
    if abs(cap-round(cap)) <= 1e-7: cap=float(round(cap))
    if cap < 1.0-TOL_FECHAMENTO_M1: return 0.0
    lim=float(max(0.0,row.get('LIMIT_PCS',0) or 0))
    lote=float(max(0.0,row.get('LOTE_MIN',0) or 0))
    emb=float(max(0.0,row.get('QTD_EMB',0) or 0))
    tipo=str(row.get('TIPO_PROD','')).strip().upper()
    flag_lote, flag_emb = _flags_operacionais_m1()
    minimo=max(1.0,float(np.ceil(lim)))
    if flag_lote and lote>0: minimo=max(minimo,float(np.ceil(lote)))
    if flag_emb and tipo in {'PA','MR'} and emb>0:
        minimo=float(np.ceil(minimo/emb)*emb)
        q=float(np.floor(cap/emb)*emb)
    else:
        q=float(np.floor(cap+TOL_FECHAMENTO_M1))
    return q if q+TOL_FECHAMENTO_M1>=minimo else 0.0

def _montar_grafo_m1(estrutura_base, nec_base):
    """Monta o grafo estrutural uma vez.

    A coluna UNID_PROD da estrutura explodida representa a chave de retorno da necessidade.
    Para achar o filho no plano, a regra usada é:
        estrutura.COD_INSUMO = filho.COD_PROD
        estrutura.UNID_PROD  = filho.UNID_FAT
    Quando não houver esse casamento, usa fallback por COD_PROD + UNID_PROD.
    """
    est = estrutura_base.copy()
    if est.empty:
        return est, pd.DataFrame(), {}, {}
    est['MES_REF'] = pd.to_datetime(est['MES_REF'], errors='coerce')
    for c in ['ID_PROD_UNID_FAT_ORIGEM','COD_PROD_ORIGEM','COD_PROD_ACAB','COD_INSUMO','UNID_PROD']:
        if c in est.columns:
            est[c] = est[c].astype(str)
    for c in ['NEC_RAIZ_PCS','NEC_GERADA_BRUTA_PCS','ESTOQUE_CONSUMIDO_PCS','NEC_GERADA_LIQ_PCS','QTD_UTIL_PCS','NIVEL']:
        if c in est.columns:
            est[c] = _num_m1(est[c])

    nec = nec_base.copy()
    nec['MES_REF'] = pd.to_datetime(nec['MES_REF'], errors='coerce')
    for c in ['ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT']:
        if c in nec.columns:
            nec[c] = nec[c].astype(str)

    mapa_fat = (
        nec[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT']]
        .drop_duplicates(['MES_REF','COD_PROD','UNID_FAT'])
        .rename(columns={'ID_PROD_UNID_FAT':'ID_FILHO','UNID_PROD':'UNID_PROD_FILHO','UNID_FAT':'UNID_FAT_FILHO'})
    )
    grafo = est.merge(
        mapa_fat,
        left_on=['MES_REF','COD_INSUMO','UNID_PROD'],
        right_on=['MES_REF','COD_PROD','UNID_FAT_FILHO'],
        how='left'
    )

    sem = grafo['ID_FILHO'].isna()
    if sem.any():
        mapa_prod = (
            nec[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT']]
            .drop_duplicates(['MES_REF','COD_PROD','UNID_PROD'])
            .rename(columns={'ID_PROD_UNID_FAT':'ID_FILHO_FB','UNID_PROD':'UNID_PROD_FILHO_FB','UNID_FAT':'UNID_FAT_FILHO_FB'})
        )
        fb = grafo.loc[sem, ['MES_REF','COD_INSUMO','UNID_PROD']].merge(
            mapa_prod,
            left_on=['MES_REF','COD_INSUMO','UNID_PROD'],
            right_on=['MES_REF','COD_PROD','UNID_PROD_FILHO_FB'],
            how='left'
        )
        grafo.loc[sem, 'ID_FILHO'] = fb['ID_FILHO_FB'].values
        grafo.loc[sem, 'UNID_PROD_FILHO'] = fb['UNID_PROD_FILHO_FB'].values
        grafo.loc[sem, 'UNID_FAT_FILHO'] = fb['UNID_FAT_FILHO_FB'].values

    grafo = grafo[grafo['ID_FILHO'].notna()].copy()
    grafo['ID_FILHO'] = grafo['ID_FILHO'].astype(str)

    # Uma aresta consolidada por raiz -> filho. O grafo completo ainda preserva as linhas para exportação.
    arestas = (
        grafo.groupby(['MES_REF','ID_PROD_UNID_FAT_ORIGEM','ID_FILHO'], as_index=False)
        .agg(
            COD_PROD_ORIGEM=('COD_PROD_ORIGEM','first'),
            COD_INSUMO=('COD_INSUMO','first'),
            UNID_RETORNO=('UNID_PROD','first'),
            NEC_RAIZ_ORIGINAL=('NEC_RAIZ_PCS','max'),
            BRUTA_ORIGINAL=('NEC_GERADA_BRUTA_PCS','sum'),
            ESTOQUE_ORIGINAL=('ESTOQUE_CONSUMIDO_PCS','sum'),
            LIQUIDA_ORIGINAL=('NEC_GERADA_LIQ_PCS','sum'),
            MENOR_NIVEL=('NIVEL','min')
        )
    )
    arestas = arestas[arestas['NEC_RAIZ_ORIGINAL'] > TOL_FECHAMENTO_M1].copy()
    filhos_por_pai = defaultdict(set)
    pais_por_filho = defaultdict(set)
    for r in arestas.itertuples(index=False):
        pai = (r.MES_REF, r.ID_PROD_UNID_FAT_ORIGEM)
        filho = (r.MES_REF, r.ID_FILHO)
        filhos_por_pai[pai].add(filho)
        pais_por_filho[filho].add(pai)
    return grafo, arestas, filhos_por_pai, pais_por_filho


def _estoque_final_por_filho_m1(arestas, prod_item, meta):
    # REGRA MESTRE: estoque fisico integro e unico, aplicado uma vez sobre demanda direta bruta + estrutural bruta.
    prod_map = prod_item.set_index(['MES_REF','ID_PROD_UNID_FAT'])['QTD_ITEM'] if not prod_item.empty else pd.Series(dtype=float)
    a = arestas.copy()
    if a.empty:
        return a, pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','NEC_COMP_FINAL','NEC_BRUTA_FINAL','ESTOQUE_FINAL','DEMANDA_BRUTA_TOTAL','NEC_LIQ_TOTAL','ATENDIMENTO_TOTAL_VALIDO','FATOR_ATEND_TOTAL'])
    a['QTD_PAI_ATUAL']=[float(prod_map.get((r.MES_REF,r.ID_PROD_UNID_FAT_ORIGEM),0.0)) for r in a.itertuples(index=False)]
    a['FATOR_PAI_ATUAL']=np.where(a['NEC_RAIZ_ORIGINAL']>TOL_FECHAMENTO_M1,a['QTD_PAI_ATUAL']/a['NEC_RAIZ_ORIGINAL'],0.0)
    a['FATOR_PAI_ATUAL']=a['FATOR_PAI_ATUAL'].clip(lower=0.0)
    a['NEC_BRUTA_FINAL_ARESTA']=(a['BRUTA_ORIGINAL']*a['FATOR_PAI_ATUAL']).clip(lower=0.0)
    estrut=(a.groupby(['MES_REF','ID_FILHO'],as_index=False).agg(NEC_BRUTA_FINAL=('NEC_BRUTA_FINAL_ARESTA','sum')).rename(columns={'ID_FILHO':'ID_PROD_UNID_FAT'}))
    cols=['MES_REF','ID_PROD_UNID_FAT','NEC_DIRETA_BRUTA_PCS','ESTOQUE_FISICO_ORIGEM','LIMIT_PCS','LOTE_MIN','QTD_EMB','TIPO_PROD']
    m=meta[[c for c in cols if c in meta.columns]].copy()
    comp=m.merge(estrut,on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    comp['NEC_BRUTA_FINAL']=_num_m1(comp.get('NEC_BRUTA_FINAL',0))
    comp['NEC_DIRETA_BRUTA_PCS']=_num_m1(comp.get('NEC_DIRETA_BRUTA_PCS',0))
    comp['ESTOQUE_FISICO_ORIGEM']=_num_m1(comp.get('ESTOQUE_FISICO_ORIGEM',0)).clip(lower=0.0)
    comp['DEMANDA_BRUTA_TOTAL']=(comp['NEC_DIRETA_BRUTA_PCS']+comp['NEC_BRUTA_FINAL']).clip(lower=0.0)
    comp['ESTOQUE_FINAL']=np.minimum(comp['DEMANDA_BRUTA_TOTAL'],comp['ESTOQUE_FISICO_ORIGEM'])
    comp['NEC_LIQ_TOTAL']=(comp['DEMANDA_BRUTA_TOTAL']-comp['ESTOQUE_FINAL']).clip(lower=0.0)
    # Parcela estrutural liquida e parcela direta liquida por rateio proporcional do atendimento/estoque compartilhado.
    ratio_liq=np.where(comp['DEMANDA_BRUTA_TOTAL']>TOL_FECHAMENTO_M1,comp['NEC_LIQ_TOTAL']/comp['DEMANDA_BRUTA_TOTAL'],0.0)
    comp['NEC_COMP_FINAL']=comp['NEC_BRUTA_FINAL']*ratio_liq
    comp['NEC_DIRETA_FINAL']=comp['NEC_DIRETA_BRUTA_PCS']*ratio_liq
    # Quantidade operacional alvo pode exceder a liquida por minimo/multiplo; excesso nao vira suporte estrutural.
    comp['QTD_OPERACIONAL_ALVO']=[_quantidade_operacional_alvo_m1(r.NEC_LIQ_TOTAL,r._asdict()) for r in comp.itertuples(index=False)]
    prod_f=[float(prod_map.get((r.MES_REF,r.ID_PROD_UNID_FAT),0.0)) for r in comp.itertuples(index=False)]
    comp['QTD_PROD_ATUAL']=prod_f
    comp['PRODUCAO_UTIL_PARA_ATENDIMENTO']=np.minimum(comp['QTD_PROD_ATUAL'].clip(lower=0.0),comp['NEC_LIQ_TOTAL'])
    comp['ATENDIMENTO_TOTAL_VALIDO']=np.minimum(comp['DEMANDA_BRUTA_TOTAL'],comp['ESTOQUE_FINAL']+comp['PRODUCAO_UTIL_PARA_ATENDIMENTO'])
    comp['FATOR_ATEND_TOTAL']=np.where(comp['DEMANDA_BRUTA_TOTAL']>TOL_FECHAMENTO_M1,comp['ATENDIMENTO_TOTAL_VALIDO']/comp['DEMANDA_BRUTA_TOTAL'],1.0).clip(0.0,1.0)
    a=a.merge(comp[['MES_REF','ID_PROD_UNID_FAT','NEC_BRUTA_FINAL','ESTOQUE_FINAL','NEC_COMP_FINAL','DEMANDA_BRUTA_TOTAL','FATOR_ATEND_TOTAL']].rename(columns={'ID_PROD_UNID_FAT':'ID_FILHO'}),on=['MES_REF','ID_FILHO'],how='left')
    a['PESO_BRUTA_FILHO']=np.where(a['NEC_BRUTA_FINAL']>TOL_FECHAMENTO_M1,a['NEC_BRUTA_FINAL_ARESTA']/a['NEC_BRUTA_FINAL'],0.0)
    # Atribuicao proporcional do estoque estrutural apenas para linhagem/auditoria.
    estoque_estrut_total=np.minimum(a['NEC_BRUTA_FINAL'],a['ESTOQUE_FINAL']*np.where(a['DEMANDA_BRUTA_TOTAL']>TOL_FECHAMENTO_M1,a['NEC_BRUTA_FINAL']/a['DEMANDA_BRUTA_TOTAL'],0.0))
    a['ESTOQUE_FINAL_ARESTA']=estoque_estrut_total*a['PESO_BRUTA_FILHO']
    a['NEC_LIQ_FINAL_ARESTA']=(a['NEC_BRUTA_FINAL_ARESTA']-a['ESTOQUE_FINAL_ARESTA']).clip(lower=0.0)
    return a, comp

def _aplicar_alvo_item_m1(pl, alvos, motivo):
    # Corta para quantidade operacional valida. Distribui deterministicamente pelas rotas atuais sem fracionar pecas.
    pl=pl.copy(); liberada=[]
    if alvos is None or alvos.empty: return pl,pd.DataFrame(liberada)
    alvos=alvos.copy(); alvos['MES_REF']=pd.to_datetime(alvos['MES_REF'],errors='coerce'); alvos['QTD_ALVO']=_num_m1(alvos['QTD_ALVO']).clip(lower=0.0)
    atual=_prod_item_m1(pl); chk=atual.merge(alvos[['MES_REF','ID_PROD_UNID_FAT','QTD_ALVO']],on=['MES_REF','ID_PROD_UNID_FAT'],how='inner')
    chk=chk[(chk['QTD_ITEM']-chk['QTD_ALVO'])>TOL_FECHAMENTO_M1]
    for r in chk.itertuples(index=False):
        mask=(pl['MES_REF'].eq(r.MES_REF))&(pl['ID_PROD_UNID_FAT'].eq(r.ID_PROD_UNID_FAT))&(_num_m1(pl['QTD_PRODUZIR_SOLVER'])>TOL_FECHAMENTO_M1)
        idx=pl.index[mask].tolist()
        if not idx: continue
        idx=sorted(idx,key=lambda i:(float(pl.at[i,'PRIOR_MATPAR']) if 'PRIOR_MATPAR' in pl.columns else 0,float(pl.at[i,'PRIOR_ROT']) if 'PRIOR_ROT' in pl.columns else 0,str(pl.at[i,'ALOC_REC']) if 'ALOC_REC' in pl.columns else '',str(pl.at[i,'COD_FER_UNID']) if 'COD_FER_UNID' in pl.columns else ''))
        restante=int(round(float(r.QTD_ALVO)))
        # Mantem producao nas rotas prioritarias ate o limite que ja existia nelas.
        novas={}
        for ix in idx:
            ant=float(pl.at[ix,'QTD_PRODUZIR_SOLVER'] or 0)
            q=min(int(np.floor(ant+TOL_FECHAMENTO_M1)),restante)
            novas[ix]=float(q); restante-=q
        if restante>0 and idx: novas[idx[-1]]+=float(restante)
        for ix in idx:
            qtd_ant=float(pl.at[ix,'QTD_PRODUZIR_SOLVER'] or 0); qtd_nova=novas.get(ix,0.0)
            pcs=float(pl.at[ix,'PCS_HORA'] or 0) if 'PCS_HORA' in pl.columns else 0.0
            hr_ant=float(pl.at[ix,'HR_PRODUZIR_SOLVER'] or 0) if 'HR_PRODUZIR_SOLVER' in pl.columns else 0.0
            hr_nova=qtd_nova/pcs if pcs>TOL_FECHAMENTO_M1 else 0.0
            qtd_c=max(0.0,qtd_ant-qtd_nova); hr_lib=max(0.0,hr_ant-hr_nova)
            pl.at[ix,'QTD_PRODUZIR_SOLVER']=qtd_nova
            if 'HR_PRODUZIR_SOLVER' in pl.columns: pl.at[ix,'HR_PRODUZIR_SOLVER']=hr_nova
            if 'QTD_REDUCAO_POS_PAI_FILHO' in pl.columns: pl.at[ix,'QTD_REDUCAO_POS_PAI_FILHO']=float(pl.at[ix,'QTD_REDUCAO_POS_PAI_FILHO'] or 0)+qtd_c
            if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in pl.columns: pl.at[ix,'QTD_PRODUZIR_FINAL_VIAVEL_EST']=qtd_nova
            if 'QTD_FINAL_APOS_REALOCACAO' in pl.columns: pl.at[ix,'QTD_FINAL_APOS_REALOCACAO']=qtd_nova
            if 'HR_LIBERADA_EST' in pl.columns: pl.at[ix,'HR_LIBERADA_EST']=float(pl.at[ix,'HR_LIBERADA_EST'] or 0)+hr_lib
            if 'MOTIVO_AJUSTE_PAI_FILHO' in pl.columns: pl.at[ix,'MOTIVO_AJUSTE_PAI_FILHO']=motivo
            if qtd_c>TOL_FECHAMENTO_M1 or hr_lib>TOL_FECHAMENTO_M1:
                liberada.append({'MES_REF':r.MES_REF,'ID_PROD_UNID_FAT':r.ID_PROD_UNID_FAT,'COD_PROD':pl.at[ix,'COD_PROD'] if 'COD_PROD' in pl.columns else '','UNID_PROD':pl.at[ix,'UNID_PROD'] if 'UNID_PROD' in pl.columns else '','ALOC_REC':pl.at[ix,'ALOC_REC'] if 'ALOC_REC' in pl.columns else '','COD_FER_UNID':pl.at[ix,'COD_FER_UNID'] if 'COD_FER_UNID' in pl.columns else '','QTD_CORTADA':qtd_c,'HR_LIBERADA':hr_lib,'MOTIVO_LIBERACAO':motivo})
    return pl,pd.DataFrame(liberada)

def _realocar_m1(pl, demanda_final, capacidade_liberada=None, itens_bloqueados=None):
    # Balanceamento operacional por blocos validos usando saldo real de maquina e ferramenta.
    # : evita recomputar o agrupamento inteiro do plano a cada alternativa.
    pl=pl.copy(); log=[]
    bloqueados=set(str(x) for x in (itens_bloqueados or set()))
    if pl.empty: return pl,pd.DataFrame(log)
    dem=demanda_final[['MES_REF','ID_PROD_UNID_FAT','QTD_OPERACIONAL_ALVO']].copy()
    dem['MES_REF']=pd.to_datetime(dem['MES_REF'],errors='coerce')
    prod=_prod_item_m1(pl)
    gaps=dem.merge(prod,on=['MES_REF','ID_PROD_UNID_FAT'],how='left').fillna({'QTD_ITEM':0.0})
    gaps['QTD_GAP']=(gaps['QTD_OPERACIONAL_ALVO']-gaps['QTD_ITEM']).clip(lower=0.0)
    gaps=gaps[gaps['QTD_GAP']>TOL_FECHAMENTO_M1].copy()
    if gaps.empty: return pl,pd.DataFrame(log)
    meta=_meta_item_m1(); gaps=gaps.merge(meta,on=['MES_REF','ID_PROD_UNID_FAT'],how='left',suffixes=('','_META'))
    alt=bd_alternativas_validas.copy(); alt['MES_REF']=pd.to_datetime(alt['MES_REF'],errors='coerce')
    for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT','HOR_REC','HOR_FER']:
        if c in alt.columns: alt[c]=_num_m1(alt[c])
    alt=alt.sort_values([c for c in ['MES_REF','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID','ID_PROD_UNID_FAT'] if c in alt.columns])
    uso_rec=pl.groupby(['MES_REF','ALOC_REC'],as_index=False).agg(HR_USO=('HR_PRODUZIR_SOLVER','sum'))
    cap_rec=alt.groupby(['MES_REF','ALOC_REC'],as_index=False).agg(HOR_REC=('HOR_REC','max')).merge(uso_rec,on=['MES_REF','ALOC_REC'],how='left').fillna({'HR_USO':0.0})
    cap_rec['HR_DISP']=(cap_rec['HOR_REC']-cap_rec['HR_USO']).clip(lower=0.0)
    rec_disp={(r.MES_REF,str(r.ALOC_REC)):float(r.HR_DISP) for r in cap_rec.itertuples(index=False)}
    uso_fer=pl.groupby(['MES_REF','COD_FER_UNID'],as_index=False).agg(HR_USO=('HR_PRODUZIR_SOLVER','sum'))
    cap_fer=alt.groupby(['MES_REF','COD_FER_UNID'],as_index=False).agg(HOR_FER=('HOR_FER','max')).merge(uso_fer,on=['MES_REF','COD_FER_UNID'],how='left').fillna({'HR_USO':0.0})
    cap_fer['HR_DISP']=(cap_fer['HOR_FER']-cap_fer['HR_USO']).clip(lower=0.0)
    fer_disp={(r.MES_REF,str(r.COD_FER_UNID)):float(r.HR_DISP) for r in cap_fer.itertuples(index=False)}
    prod_map={(r.MES_REF,str(r.ID_PROD_UNID_FAT)):float(r.QTD_ITEM) for r in prod.itertuples(index=False)}
    gap_map={(r.MES_REF,str(r.ID_PROD_UNID_FAT)):float(r.QTD_GAP) for r in gaps.itertuples(index=False)}
    gap_row={(r.MES_REF,str(r.ID_PROD_UNID_FAT)):r._asdict() for r in gaps.itertuples(index=False)}
    rodada=0; mudou=True
    while mudou:
        mudou=False; rodada+=1
        if rodada>10000: raise RuntimeError('REALOCACAO_M1_EXCEDEU_LIMITE_DE_RODADAS')
        for a in alt.itertuples(index=False):
            item=str(getattr(a,'ID_PROD_UNID_FAT')); mes=getattr(a,'MES_REF'); rec=str(getattr(a,'ALOC_REC')); fer=str(getattr(a,'COD_FER_UNID'))
            chave=(mes,item); gap=float(gap_map.get(chave,0.0))
            if item in bloqueados: continue
            if gap<=TOL_FECHAMENTO_M1: continue
            pcs=float(getattr(a,'PCS_HORA') or 0.0)
            if pcs<=TOL_FECHAMENTO_M1: continue
            hr=min(float(rec_disp.get((mes,rec),0.0)),float(fer_disp.get((mes,fer),0.0)))
            if hr<=TOL_FECHAMENTO_M1: continue
            atual=float(prod_map.get(chave,0.0)); teto=atual+min(gap,hr*pcs)
            alvo=_quantidade_valida_abaixo_m1(teto,gap_row.get(chave,{}))
            qtd_add=max(0.0,alvo-atual)
            if qtd_add<=TOL_FECHAMENTO_M1: continue
            hr_add=qtd_add/pcs
            if hr_add>hr+1e-8: continue
            mask=(pl['MES_REF'].eq(mes))&(pl['ID_PROD_UNID_FAT'].astype(str).eq(item))&(pl['ALOC_REC'].astype(str).eq(rec))&(pl['COD_FER_UNID'].astype(str).eq(fer))
            idx=pl.index[mask].tolist()
            if idx:
                ix=idx[0]
                pl.at[ix,'QTD_PRODUZIR_SOLVER']=float(pl.at[ix,'QTD_PRODUZIR_SOLVER'] or 0)+qtd_add
                pl.at[ix,'HR_PRODUZIR_SOLVER']=float(pl.at[ix,'HR_PRODUZIR_SOLVER'] or 0)+hr_add
                if 'QTD_REALOCADA_CAP_LIBERADA' in pl.columns: pl.at[ix,'QTD_REALOCADA_CAP_LIBERADA']=float(pl.at[ix,'QTD_REALOCADA_CAP_LIBERADA'] or 0)+qtd_add
                if 'QTD_FINAL_APOS_REALOCACAO' in pl.columns: pl.at[ix,'QTD_FINAL_APOS_REALOCACAO']=float(pl.at[ix,'QTD_PRODUZIR_SOLVER'])
            else:
                nova={c:0 for c in pl.columns}; ad=a._asdict()
                for c in pl.columns:
                    if c in ad: nova[c]=ad[c]
                nova.update({'MES_REF':mes,'ID_PROD_UNID_FAT':item,'QTD_PRODUZIR_SOLVER':qtd_add,'HR_PRODUZIR_SOLVER':hr_add,'STATUS_DECISAO':'ALOCADO_POR_REALOCACAO_CAP_LIBERADA','MOTIVO_STATUS':'PRODUCAO_ALOCADA_PELO_SOLVER'})
                if 'QTD_REALOCADA_CAP_LIBERADA' in pl.columns: nova['QTD_REALOCADA_CAP_LIBERADA']=qtd_add
                if 'QTD_FINAL_APOS_REALOCACAO' in pl.columns: nova['QTD_FINAL_APOS_REALOCACAO']=qtd_add
                pl=pd.concat([pl,pd.DataFrame([nova])],ignore_index=True)
            rec_disp[(mes,rec)]=max(0.0,rec_disp[(mes,rec)]-hr_add); fer_disp[(mes,fer)]=max(0.0,fer_disp[(mes,fer)]-hr_add)
            prod_map[chave]=atual+qtd_add; gap_map[chave]=max(0.0,gap-qtd_add)
            log.append({'RODADA_REALOC':rodada,'MES_REF':mes,'ID_PROD_UNID_FAT':item,'COD_PROD':getattr(a,'COD_PROD',''),'UNID_PROD':getattr(a,'UNID_PROD',''),'UNID_FAT':getattr(a,'UNID_FAT',''),'ALOC_REC':rec,'COD_FER_UNID':fer,'PCS_HORA':pcs,'QTD_REALOCADA':qtd_add,'HR_CONSUMIDA_REALOC':hr_add,'STATUS_REALOCACAO':'REALOCADO_BLOCO_VALIDO_M1'})
            mudou=True
    return pl,pd.DataFrame(log)

def _corrigir_estouro_capacidade_m1(pl, meta):
    # Blindagem operacional: nenhum bloco inteiro/multiplo pode deixar recurso ou ferramenta acima da capacidade.
    pl=pl.copy(); cortes=[]
    mm=meta.set_index(['MES_REF','ID_PROD_UNID_FAT'])
    for _pass in range(100):
        alterou=False
        # Recurso
        rr=pl.groupby(['MES_REF','ALOC_REC'],as_index=False).agg(HR_USO=('HR_PRODUZIR_SOLVER','sum'),HOR_CAP=('HOR_REC','max'))
        rr['OVER']=(rr['HR_USO']-rr['HOR_CAP']).clip(lower=0.0)
        overs=[('REC',r.MES_REF,str(r.ALOC_REC),float(r.OVER)) for r in rr[rr['OVER']>1e-10].itertuples(index=False)]
        ff=pl.groupby(['MES_REF','COD_FER_UNID'],as_index=False).agg(HR_USO=('HR_PRODUZIR_SOLVER','sum'),HOR_CAP=('HOR_FER','max'))
        ff['OVER']=(ff['HR_USO']-ff['HOR_CAP']).clip(lower=0.0)
        overs += [('FER',r.MES_REF,str(r.COD_FER_UNID),float(r.OVER)) for r in ff[ff['OVER']>1e-10].itertuples(index=False)]
        if not overs: break
        for tipo,mes,chave,over in overs:
            col='ALOC_REC' if tipo=='REC' else 'COD_FER_UNID'
            mask=(pl['MES_REF'].eq(mes))&(pl[col].astype(str).eq(chave))&(_num_m1(pl['QTD_PRODUZIR_SOLVER'])>TOL_FECHAMENTO_M1)
            cand=pl.loc[mask].copy()
            if cand.empty: continue
            sortcols=[c for c in ['PRIOR_MATPAR','PRIOR_ROT'] if c in cand.columns]
            if sortcols: cand=cand.sort_values(sortcols,ascending=False)
            feito=False
            prod_map=_prod_item_m1(pl).set_index(['MES_REF','ID_PROD_UNID_FAT'])['QTD_ITEM']
            for ix,rowline in cand.iterrows():
                pcs=float(rowline.get('PCS_HORA',0) or 0)
                if pcs<=TOL_FECHAMENTO_M1: continue
                item=str(rowline['ID_PROD_UNID_FAT']); qtot=float(prod_map.get((mes,item),0.0)); qlinha=float(rowline.get('QTD_PRODUZIR_SOLVER',0) or 0)
                if qtot<=0 or qlinha<=0: continue
                try: mrow=mm.loc[(mes,item)].to_dict()
                except Exception: mrow={}
                qmin_cut=max(1.0,np.ceil(over*pcs-1e-12))
                alvo=_quantidade_valida_abaixo_m1(max(0.0,qtot-qmin_cut),mrow)
                corte=max(0.0,qtot-alvo)
                if corte<=TOL_FECHAMENTO_M1 or corte>qlinha+TOL_FECHAMENTO_M1: continue
                qtd_nova=max(0.0,qlinha-corte); hr_ant=float(rowline.get('HR_PRODUZIR_SOLVER',0) or 0); hr_nova=qtd_nova/pcs
                pl.at[ix,'QTD_PRODUZIR_SOLVER']=qtd_nova; pl.at[ix,'HR_PRODUZIR_SOLVER']=hr_nova
                if 'QTD_REDUCAO_POS_PAI_FILHO' in pl.columns: pl.at[ix,'QTD_REDUCAO_POS_PAI_FILHO']=float(pl.at[ix,'QTD_REDUCAO_POS_PAI_FILHO'] or 0)+corte
                if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in pl.columns: pl.at[ix,'QTD_PRODUZIR_FINAL_VIAVEL_EST']=qtd_nova
                if 'QTD_FINAL_APOS_REALOCACAO' in pl.columns: pl.at[ix,'QTD_FINAL_APOS_REALOCACAO']=qtd_nova
                if 'HR_LIBERADA_EST' in pl.columns: pl.at[ix,'HR_LIBERADA_EST']=float(pl.at[ix,'HR_LIBERADA_EST'] or 0)+max(0.0,hr_ant-hr_nova)
                if 'MOTIVO_AJUSTE_PAI_FILHO' in pl.columns: pl.at[ix,'MOTIVO_AJUSTE_PAI_FILHO']='AJUSTE_CAPACIDADE_BLOCO_VALIDO_M1'
                cortes.append({'MES_REF':mes,'ID_PROD_UNID_FAT':item,'COD_PROD':rowline.get('COD_PROD',''),'UNID_PROD':rowline.get('UNID_PROD',''),'ALOC_REC':rowline.get('ALOC_REC',''),'COD_FER_UNID':rowline.get('COD_FER_UNID',''),'QTD_CORTADA':corte,'HR_LIBERADA':max(0.0,hr_ant-hr_nova),'MOTIVO_LIBERACAO':'AJUSTE_CAPACIDADE_BLOCO_VALIDO_M1'})
                alterou=True; feito=True; break
            if feito: break
        if not alterou: break
    return pl,pd.DataFrame(cortes)

def _demanda_final_m1(comp_final, meta):
    cols=['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_DIRETA_PCS','NEC_DIRETA_BRUTA_PCS','LIMIT_PCS','LOTE_MIN','QTD_EMB']
    dem=meta[[c for c in cols if c in meta.columns]].copy()
    ccols=['MES_REF','ID_PROD_UNID_FAT','NEC_COMP_FINAL','NEC_BRUTA_FINAL','ESTOQUE_FINAL','NEC_LIQ_TOTAL','NEC_DIRETA_FINAL','QTD_OPERACIONAL_ALVO','DEMANDA_BRUTA_TOTAL','FATOR_ATEND_TOTAL']
    dem=dem.merge(comp_final[[c for c in ccols if c in comp_final.columns]],on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    for c in ['NEC_COMP_FINAL','NEC_BRUTA_FINAL','ESTOQUE_FINAL','NEC_LIQ_TOTAL','NEC_DIRETA_FINAL','QTD_OPERACIONAL_ALVO','DEMANDA_BRUTA_TOTAL','FATOR_ATEND_TOTAL']:
        if c not in dem.columns: dem[c]=0.0
        dem[c]=_num_m1(dem[c])
    # Para itens sem estrutura, manter necessidade direta liquida original como base; para itens compartilhados, usar liquida pooled.
    sem_pool=dem['DEMANDA_BRUTA_TOTAL']<=TOL_FECHAMENTO_M1
    base_direta=_num_m1(dem.get('NEC_DIRETA_PCS',0))
    dem.loc[sem_pool,'NEC_LIQ_TOTAL']=base_direta[sem_pool]
    dem.loc[sem_pool,'QTD_OPERACIONAL_ALVO']=[_quantidade_operacional_alvo_m1(q,r._asdict()) for q,r in zip(dem.loc[sem_pool,'NEC_LIQ_TOTAL'],dem.loc[sem_pool].itertuples(index=False))]
    dem['QTD_DEMANDA_FINAL']=dem['NEC_LIQ_TOTAL'].clip(lower=0.0)
    return dem

def _limite_pai_por_corte_solver_m1(arestas_calc, comp_final, prod_item, meta, gap_direto_atribuido):
    # O pai só pode ser reduzido pelo gap REAL de produção do componente.
    # Queda da NEC do componente provocada por estoque ou reexplosão não entra neste cálculo.
    if arestas_calc.empty or comp_final.empty:
        return pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_ALVO']), gap_direto_atribuido, pd.DataFrame()
    comp=comp_final.copy()
    comp['CORTE_SOLVER_TOTAL']=(comp['NEC_LIQ_TOTAL']-comp['PRODUCAO_UTIL_PARA_ATENDIMENTO']).clip(lower=0.0)
    prod_map={(r.MES_REF,str(r.ID_PROD_UNID_FAT)):float(r.QTD_ITEM) for r in prod_item.itertuples(index=False)}
    mm=meta.set_index(['MES_REF','ID_PROD_UNID_FAT'])
    locked=dict(gap_direto_atribuido or {})
    cortes=[]; auditoria=[]
    arestas_by_child={k:g.copy() for k,g in arestas_calc.groupby(['MES_REF','ID_FILHO'])}
    for c in comp.itertuples(index=False):
        chave_filho=(c.MES_REF,str(c.ID_PROD_UNID_FAT))
        corte_total=float(max(0.0,c.CORTE_SOLVER_TOTAL))
        locked_prev=min(float(max(0.0,locked.get(chave_filho,0.0))),corte_total)
        corte_novo=max(0.0,corte_total-locked_prev)
        if corte_novo<=TOL_FECHAMENTO_M1:
            locked[chave_filho]=locked_prev; continue
        demanda_total=float(max(0.0,c.DEMANDA_BRUTA_TOTAL)); direta_bruta=float(max(0.0,c.NEC_DIRETA_BRUTA_PCS))
        if demanda_total<=TOL_FECHAMENTO_M1:
            locked[chave_filho]=locked_prev; continue
        corte_direto_novo=corte_novo*(direta_bruta/demanda_total)
        locked[chave_filho]=locked_prev+corte_direto_novo
        g=arestas_by_child.get(chave_filho)
        if g is None or g.empty: continue
        for e in g.itertuples(index=False):
            bruta_aresta=float(max(0.0,e.NEC_BRUTA_FINAL_ARESTA)); pai_chave=(e.MES_REF,str(e.ID_PROD_UNID_FAT_ORIGEM)); qtd_pai=float(max(0.0,prod_map.get(pai_chave,0.0)))
            if bruta_aresta<=TOL_FECHAMENTO_M1 or qtd_pai<=TOL_FECHAMENTO_M1: continue
            corte_estrutura=corte_novo*(bruta_aresta/demanda_total)
            # Limita a parcela propagada à demanda estrutural efetivamente ativa da própria origem.
            corte_estrutura=min(max(0.0,corte_estrutura),bruta_aresta)
            if corte_estrutura<=TOL_FECHAMENTO_M1: continue
            coef=bruta_aresta/qtd_pai
            if coef<=TOL_FECHAMENTO_M1: continue
            cap=max(0.0,qtd_pai-(corte_estrutura/coef))
            try: row=mm.loc[(e.MES_REF,str(e.ID_PROD_UNID_FAT_ORIGEM))].to_dict()
            except Exception: row={}
            alvo=_quantidade_valida_abaixo_m1(cap,row)
            cortes.append({'MES_REF':e.MES_REF,'ID_PROD_UNID_FAT':str(e.ID_PROD_UNID_FAT_ORIGEM),'QTD_ALVO':alvo})
            auditoria.append({'MES_REF':e.MES_REF,'ID_FILHO':str(e.ID_FILHO),'ID_PAI':str(e.ID_PROD_UNID_FAT_ORIGEM),'CORTE_SOLVER_TOTAL':corte_total,'CORTE_NOVO_PROPAGADO':corte_novo,'CORTE_DIRETO_NOVO':corte_direto_novo,'CORTE_ESTRUTURAL_ARESTA':corte_estrutura,'QTD_PAI_ANTES':qtd_pai,'QTD_PAI_ALVO':alvo})
    if not cortes:
        return pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_ALVO']), locked, pd.DataFrame(auditoria)
    lim=pd.DataFrame(cortes).groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False).agg(QTD_ALVO=('QTD_ALVO','min'))
    return lim, locked, pd.DataFrame(auditoria)

def _materializar_estrutura_final_m1(grafo_base, arestas_calc, comp_final):
    if grafo_base.empty: return grafo_base.copy()
    fatores=arestas_calc[['MES_REF','ID_PROD_UNID_FAT_ORIGEM','ID_FILHO','FATOR_PAI_ATUAL']].copy()
    g=grafo_base.copy().merge(fatores,on=['MES_REF','ID_PROD_UNID_FAT_ORIGEM','ID_FILHO'],how='left')
    g['FATOR_PAI_ATUAL']=_num_m1(g['FATOR_PAI_ATUAL']); g['NEC_RAIZ_PCS']=_num_m1(g['NEC_RAIZ_PCS'])*g['FATOR_PAI_ATUAL']; g['NEC_GERADA_BRUTA_PCS']=_num_m1(g['NEC_GERADA_BRUTA_PCS'])*g['FATOR_PAI_ATUAL']
    c=comp_final[['MES_REF','ID_PROD_UNID_FAT','ESTOQUE_FINAL','DEMANDA_BRUTA_TOTAL','NEC_BRUTA_FINAL']].rename(columns={'ID_PROD_UNID_FAT':'ID_FILHO'})
    g=g.merge(c,on=['MES_REF','ID_FILHO'],how='left')
    # Estoque total e unico; parcela atribuida a estrutura proporcional ao peso estrutural dentro da demanda bruta total.
    est_estr=np.minimum(_num_m1(g['NEC_BRUTA_FINAL']), _num_m1(g['ESTOQUE_FINAL'])*np.where(_num_m1(g['DEMANDA_BRUTA_TOTAL'])>TOL_FECHAMENTO_M1,_num_m1(g['NEC_BRUTA_FINAL'])/_num_m1(g['DEMANDA_BRUTA_TOTAL']),0.0))
    total_bruta=g.groupby(['MES_REF','ID_FILHO'])['NEC_GERADA_BRUTA_PCS'].transform('sum')
    peso=np.where(total_bruta>TOL_FECHAMENTO_M1,g['NEC_GERADA_BRUTA_PCS']/total_bruta,0.0)
    # est_estr se repete por linha; obter valor por filho com transform first
    est_estr_series=pd.Series(est_estr,index=g.index); est_estr_unico=est_estr_series.groupby([g['MES_REF'],g['ID_FILHO']]).transform('first')
    g['ESTOQUE_CONSUMIDO_PCS']=est_estr_unico*peso; g['NEC_GERADA_LIQ_PCS']=(g['NEC_GERADA_BRUTA_PCS']-g['ESTOQUE_CONSUMIDO_PCS']).clip(lower=0.0)
    if 'ESTOQUE_COMP_ANTES_PCS' in g.columns: g['ESTOQUE_COMP_ANTES_PCS']=_num_m1(g['ESTOQUE_FINAL'])
    if 'ESTOQUE_COMP_DEPOIS_PCS' in g.columns: g['ESTOQUE_COMP_DEPOIS_PCS']=(_num_m1(g['ESTOQUE_FINAL'])-g['ESTOQUE_CONSUMIDO_PCS']).clip(lower=0.0)
    g=g[g['NEC_GERADA_BRUTA_PCS']>TOL_FECHAMENTO_M1].copy()
    drop=[c for c in ['ID_FILHO','UNID_PROD_FILHO','UNID_FAT_FILHO','COD_PROD','FATOR_PAI_ATUAL','ESTOQUE_FINAL','DEMANDA_BRUTA_TOTAL','NEC_BRUTA_FINAL'] if c in g.columns]
    g=g.drop(columns=drop,errors='ignore')
    return g[[c for c in estrutura_explodida.columns if c in g.columns]].copy()

_COLS_01_RECURSOS_M1 = list(resumo_recursos.columns)
_COLS_02_FERRAMENTAS_M1 = list(resumo_ferramentas.columns)
_COLS_03_PLANO_M1 = list(plano_producao.columns)
_COLS_04_NAO_ATEND_M1 = list(nao_atend_item.columns)
_COLS_12_NEC_M1 = list(nec_consolidada_solver.columns)
plano_producao = _normalizar_m1(plano_producao)
meta_m1 = _meta_item_m1()
grafo_base_m1, arestas_base_m1, filhos_por_pai_m1, pais_por_filho_m1 = _montar_grafo_m1(estrutura_explodida, meta_m1)
ids_estrutura_m1 = set(arestas_base_m1['ID_PROD_UNID_FAT_ORIGEM'].astype(str)).union(set(arestas_base_m1['ID_FILHO'].astype(str)))

# Guarda chaves impactadas apenas para rastreio; o cálculo é vetorizado no grafo pré-calculado.
fila_down_m1 = deque(filhos_por_pai_m1.keys())
fila_up_m1 = deque(pais_por_filho_m1.keys())
logs_lib_m1 = []
logs_realoc_m1 = []
iter_executadas_m1 = 0

#  - normaliza o plano inicial para pecas inteiras e blocos operacionais validos antes do fechamento.
_prod_ini_oper_m1 = _prod_item_m1(plano_producao).merge(meta_m1, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
_alvos_ini_oper_m1 = []
for _r in _prod_ini_oper_m1.itertuples(index=False):
    _row = _r._asdict(); _q = float(_r.QTD_ITEM or 0.0)
    _alvos_ini_oper_m1.append({'MES_REF':_r.MES_REF,'ID_PROD_UNID_FAT':str(_r.ID_PROD_UNID_FAT),'QTD_ALVO':_quantidade_valida_abaixo_m1(_q,_row)})
plano_producao, _lib_oper_ini_m1 = _aplicar_alvo_item_m1(plano_producao, pd.DataFrame(_alvos_ini_oper_m1), 'AJUSTE_QUANTIDADE_OPERACIONAL_INICIAL_M1')
if not _lib_oper_ini_m1.empty:
    _lib_oper_ini_m1['ITERACAO_FECHAMENTO'] = 0
    logs_lib_m1.append(_lib_oper_ini_m1)

prod_ant_m1 = _prod_item_m1(plano_producao)
estados_m1 = set()
convergiu_m1 = False

gap_direto_atribuido = {}
logs_corte_solver_m1 = []

for iter_m1 in range(1, MAX_ITER_FECHAMENTO_M1 + 1):
    iter_executadas_m1 = iter_m1
    prod_item_m1 = _prod_item_m1(plano_producao)
    arestas_calc_m1, comp_final_m1 = _estoque_final_por_filho_m1(arestas_base_m1, prod_item_m1, meta_m1)
    demanda_final_m1 = _demanda_final_m1(comp_final_m1, meta_m1)

    # DOWN: quando a produção dos pais muda, a NEC do componente é recalculada.
    # Se a nova NEC for menor, somente a produção excedente do componente é retirada.
    # Essa retirada é ajuste de necessidade e NÃO é tratada como corte de capacidade do componente.
    alvos_filhos_m1 = demanda_final_m1[['MES_REF','ID_PROD_UNID_FAT','QTD_OPERACIONAL_ALVO']].rename(columns={'QTD_OPERACIONAL_ALVO':'QTD_ALVO'})
    plano_producao, lib_down_m1 = _aplicar_alvo_item_m1(plano_producao, alvos_filhos_m1, 'CORTE_EXCESSO_FILHO_APOS_REEXPLOSAO_PAI_M1')
    if not lib_down_m1.empty:
        lib_down_m1['ITERACAO_FECHAMENTO'] = iter_m1; logs_lib_m1.append(lib_down_m1)

    # OTIMIZACAO DO COMPONENTE: tenta atender a NEC liquida antes de qualquer corte do pai.
    prod_item_m1 = _prod_item_m1(plano_producao)
    arestas_calc_m1, comp_final_m1 = _estoque_final_por_filho_m1(arestas_base_m1, prod_item_m1, meta_m1)
    demanda_final_m1 = _demanda_final_m1(comp_final_m1, meta_m1)
    # Pais que dependem de componente ainda com gap nao podem ser restaurados pela realocacao.
    # Enquanto o componente nao possui folga para atender sua propria NEC liquida, aumentar o pai recriaria a falta.
    comp_gap_m1 = comp_final_m1.copy()
    comp_gap_m1['CORTE_SOLVER_ATUAL'] = (comp_gap_m1['NEC_LIQ_TOTAL'] - comp_gap_m1['PRODUCAO_UTIL_PARA_ATENDIMENTO']).clip(lower=0.0)
    filhos_sem_folga_m1 = set((r.MES_REF,str(r.ID_PROD_UNID_FAT)) for r in comp_gap_m1[comp_gap_m1['CORTE_SOLVER_ATUAL']>TOL_FECHAMENTO_M1].itertuples(index=False))
    pais_bloqueados_m1 = set()
    for _filho in filhos_sem_folga_m1:
        for _pai in pais_por_filho_m1.get(_filho,set()):
            pais_bloqueados_m1.add(str(_pai[1]))
    if APLICAR_REALOCACAO_M1:
        plano_producao, log_realoc_m1 = _realocar_m1(plano_producao, demanda_final_m1, lib_down_m1, pais_bloqueados_m1)
        if not log_realoc_m1.empty:
            log_realoc_m1['ITERACAO_FECHAMENTO'] = iter_m1; logs_realoc_m1.append(log_realoc_m1)

    # Blindagem de capacidade apos alocacao em blocos validos.
    plano_producao, lib_cap_m1 = _corrigir_estouro_capacidade_m1(plano_producao, meta_m1)
    if not lib_cap_m1.empty:
        lib_cap_m1['ITERACAO_FECHAMENTO'] = iter_m1; logs_lib_m1.append(lib_cap_m1)

    # UP: calcula o gap real que restou no componente depois de estoque, rateio e tentativa de produção.
    # Apenas esse gap de capacidade pode retornar para as origens estruturais e reduzir pais.
    # A parcela já atribuída à demanda direta é controlada separadamente para não cortar pais de novo.
    prod_item_m1 = _prod_item_m1(plano_producao)
    arestas_calc_m1, comp_final_m1 = _estoque_final_por_filho_m1(arestas_base_m1, prod_item_m1, meta_m1)
    limites_pai_m1, gap_direto_atribuido, audit_corte_m1 = _limite_pai_por_corte_solver_m1(arestas_calc_m1, comp_final_m1, prod_item_m1, meta_m1, gap_direto_atribuido)
    if not audit_corte_m1.empty:
        audit_corte_m1['ITERACAO_FECHAMENTO'] = iter_m1; logs_corte_solver_m1.append(audit_corte_m1)
    plano_producao, lib_up_m1 = _aplicar_alvo_item_m1(plano_producao, limites_pai_m1, 'CORTE_PAI_POR_CORTE_SOLVER_COMPONENTE_M1')
    if not lib_up_m1.empty:
        lib_up_m1['ITERACAO_FECHAMENTO'] = iter_m1; logs_lib_m1.append(lib_up_m1)

    prod_novo_m1 = _prod_item_m1(plano_producao)
    delta = prod_ant_m1.merge(prod_novo_m1, on=['MES_REF','ID_PROD_UNID_FAT'], how='outer', suffixes=('_ANT','_NOVO')).fillna(0.0)
    delta['QTD_ITEM_ANT'] = _num_m1(delta['QTD_ITEM_ANT']).round(0); delta['QTD_ITEM_NOVO'] = _num_m1(delta['QTD_ITEM_NOVO']).round(0)
    delta['ALTEROU'] = delta['QTD_ITEM_NOVO'].ne(delta['QTD_ITEM_ANT'])
    qtd_alt_m1 = int(delta['ALTEROU'].sum()) if not delta.empty else 0
    estado_m1 = tuple(delta.sort_values(['MES_REF','ID_PROD_UNID_FAT'])[['MES_REF','ID_PROD_UNID_FAT','QTD_ITEM_NOVO']].astype(str).itertuples(index=False,name=None))
    repetiu_m1 = estado_m1 in estados_m1; estados_m1.add(estado_m1)
    prod_ant_m1 = prod_novo_m1.copy()
    if qtd_alt_m1 == 0:
        convergiu_m1 = True; break
    if repetiu_m1:
        raise RuntimeError('FECHAMENTO_M1_OSCILACAO_DETECTADA: estado operacional repetido antes da convergencia.')

if not convergiu_m1:
    raise RuntimeError('FECHAMENTO_M1_NAO_CONVERGIU_NO_LIMITE_DE_ITERACOES')

# Materializa resultados finais sem alterar layout.
prod_item_m1 = _prod_item_m1(plano_producao)
arestas_calc_m1, comp_final_m1 = _estoque_final_por_filho_m1(arestas_base_m1, prod_item_m1, meta_m1)
demanda_final_m1 = _demanda_final_m1(comp_final_m1, meta_m1)
estrutura_explodida = _materializar_estrutura_final_m1(grafo_base_m1, arestas_calc_m1, comp_final_m1)

nec_explodida_componentes = (
    estrutura_explodida.groupby(['MES_REF','UNID_PROD','COD_PROD_ORIGEM','COD_INSUMO'], as_index=False)
    .agg(
        NEC_EXPLODIDA_BRUTA_PCS=('NEC_GERADA_BRUTA_PCS','sum'),
        ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=('ESTOQUE_CONSUMIDO_PCS','sum'),
        NEC_EXPLODIDA_PCS=('NEC_GERADA_LIQ_PCS','sum'),
        MENOR_NIVEL=('NIVEL','min')
    )
    if not estrutura_explodida.empty else pd.DataFrame(columns=['MES_REF','UNID_PROD','COD_PROD_ORIGEM','COD_INSUMO','NEC_EXPLODIDA_BRUTA_PCS','ESTOQUE_CONSUMIDO_EXPLOSAO_PCS','NEC_EXPLODIDA_PCS','MENOR_NIVEL'])
)

# Atualiza NEC consolidada e plano preservando colunas existentes.
nec_ref_m1 = nec_consolidada_solver.copy()
nec_ref_m1['MES_REF'] = pd.to_datetime(nec_ref_m1['MES_REF'], errors='coerce')
# A guia 12 pode ter mais de uma linha técnica para a mesma chave do item.
# Para atualizar a guia 03 sem multiplicar linhas do plano, consolidamos uma linha por MES_REF + ID_PROD_UNID_FAT.
nec_ref_m1 = nec_ref_m1.sort_values([c for c in ['MES_REF','ID_PROD_UNID_FAT','UNID_PROD','UNID_FAT','COD_PROD'] if c in nec_ref_m1.columns]).drop_duplicates(['MES_REF','ID_PROD_UNID_FAT'], keep='first')
updates_m1 = demanda_final_m1.rename(columns={
    'NEC_COMP_FINAL':'NEC_COMP_PCS',
    'NEC_BRUTA_FINAL':'NEC_EXPLODIDA_BRUTA_PCS',
    'ESTOQUE_FINAL':'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS',
    'QTD_DEMANDA_FINAL':'NEC_PCS_CONSOLIDADA'
})[['MES_REF','ID_PROD_UNID_FAT','NEC_COMP_PCS','NEC_EXPLODIDA_BRUTA_PCS','ESTOQUE_CONSUMIDO_EXPLOSAO_PCS','NEC_PCS_CONSOLIDADA']]
updates_m1['NEC_EXPLODIDA_PCS'] = updates_m1['NEC_COMP_PCS']
updates_m1 = updates_m1.merge(demanda_final_m1[['MES_REF','ID_PROD_UNID_FAT','QTD_OPERACIONAL_ALVO']], on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
nec_ref_m1 = nec_ref_m1.drop(columns=[c for c in ['NEC_COMP_PCS','NEC_EXPLODIDA_BRUTA_PCS','ESTOQUE_CONSUMIDO_EXPLOSAO_PCS','NEC_EXPLODIDA_PCS','NEC_PCS_CONSOLIDADA','NEC_PCS'] if c in nec_ref_m1.columns], errors='ignore')
nec_ref_m1 = nec_ref_m1.merge(updates_m1, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
for c in ['NEC_COMP_PCS','NEC_EXPLODIDA_BRUTA_PCS','ESTOQUE_CONSUMIDO_EXPLOSAO_PCS','NEC_EXPLODIDA_PCS','NEC_PCS_CONSOLIDADA']:
    nec_ref_m1[c] = _num_m1(nec_ref_m1[c])
nec_ref_m1 = nec_ref_m1.merge(demanda_final_m1[['MES_REF','ID_PROD_UNID_FAT','QTD_OPERACIONAL_ALVO']], on=['MES_REF','ID_PROD_UNID_FAT'], how='left', suffixes=('','_DEM')) if 'QTD_OPERACIONAL_ALVO' not in nec_ref_m1.columns else nec_ref_m1
nec_ref_m1['NEC_PCS'] = _num_m1(nec_ref_m1.get('QTD_OPERACIONAL_ALVO', nec_ref_m1['NEC_PCS_CONSOLIDADA']))
nec_consolidada_solver = nec_ref_m1[[c for c in nec_consolidada_solver.columns if c in nec_ref_m1.columns] + [c for c in nec_ref_m1.columns if c not in nec_consolidada_solver.columns]].copy()
nec_consolidada_solver = nec_consolidada_solver.drop_duplicates(['MES_REF','ID_PROD_UNID_FAT'], keep='first')

nec_cols_m1 = [c for c in ['NEC_DIRETA_PCS','NEC_COMP_PCS','NEC_EXPLODIDA_BRUTA_PCS','ESTOQUE_CONSUMIDO_EXPLOSAO_PCS','NEC_EXPLODIDA_PCS','NEC_PCS_CONSOLIDADA','NEC_PCS'] if c in nec_consolidada_solver.columns]
plano_producao = plano_producao.drop(columns=[c for c in nec_cols_m1 if c in plano_producao.columns and c != 'NEC_DIRETA_PCS'], errors='ignore')
plano_producao = plano_producao.merge(nec_consolidada_solver[['MES_REF','ID_PROD_UNID_FAT'] + [c for c in nec_cols_m1 if c != 'NEC_DIRETA_PCS']], on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
for c in nec_cols_m1:
    if c in plano_producao.columns:
        plano_producao[c] = _num_m1(plano_producao[c])
if 'QTD_FINAL_APOS_REALOCACAO' in plano_producao.columns:
    plano_producao['QTD_FINAL_APOS_REALOCACAO'] = plano_producao['QTD_PRODUZIR_SOLVER']
if 'QTD_GAP_FINAL_APOS_REALOCACAO' in plano_producao.columns:
    plano_producao['QTD_GAP_FINAL_APOS_REALOCACAO'] = (plano_producao.get('NEC_PCS_CONSOLIDADA',0) - plano_producao['QTD_PRODUZIR_SOLVER']).clip(lower=0.0)

# Recalcula guias dependentes no fim, uma única vez.
_prod_final_m1 = _prod_item_m1(plano_producao).rename(columns={'QTD_ITEM':'QTD_ATENDIDA_SOLVER'})
nao_atend_base_m1 = nec_consolidada_solver.copy()
nao_atend_item = nao_atend_base_m1.merge(_prod_final_m1, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
nao_atend_item['QTD_ATENDIDA_SOLVER'] = _num_m1(nao_atend_item['QTD_ATENDIDA_SOLVER'])
nec_col_final_m1 = 'NEC_PCS' if 'NEC_PCS' in nao_atend_item.columns else 'NEC_PCS_CONSOLIDADA'
nao_atend_item['NEC_PCS'] = _num_m1(nao_atend_item[nec_col_final_m1])
nao_atend_item['QTD_NAO_ATEND_SOLVER'] = (nao_atend_item['NEC_PCS'] - nao_atend_item['QTD_ATENDIDA_SOLVER']).clip(lower=0.0)

try:
    ajuste_pai_filho, resumo_pai_filho, capacidade_liberada_pf = calcular_ajuste_pos_solver_pai_filho(
        estrutura_explodida,
        nec_consolidada_solver,
        nao_atend_item,
        plano_producao.drop(columns=[c for c in ['FATOR_AJUSTE_SOBRE_SOLVER','FATOR_FINAL_VIAVEL_ORIGEM'] if c in plano_producao.columns], errors='ignore')
    )
except Exception as e:
    print('AVISO_M1_AJUSTE_PAI_FILHO_FINAL_NAO_RECALCULADO:', e)

#  - reconcilia as guias 13/14/15 com a regra nova e com a producao final oficial.
try:
    _pf = plano_producao.copy()
    _pf['MES_REF']=pd.to_datetime(_pf['MES_REF'],errors='coerce')
    _pf['QTD_PRODUZIR_SOLVER']=_num_m1(_pf['QTD_PRODUZIR_SOLVER'])
    _pf['QTD_REDUCAO_POS_PAI_FILHO']=_num_m1(_pf.get('QTD_REDUCAO_POS_PAI_FILHO',0))
    _pf['HR_PRODUZIR_SOLVER']=_num_m1(_pf.get('HR_PRODUZIR_SOLVER',0))
    _pf['HR_LIBERADA_EST']=_num_m1(_pf.get('HR_LIBERADA_EST',0))
    _item_final = (_pf.groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False)
        .agg(QTD_FINAL=('QTD_PRODUZIR_SOLVER','sum'),QTD_REDUCAO=('QTD_REDUCAO_POS_PAI_FILHO','sum')))
    _item_final['QTD_INICIAL_EST']=_item_final['QTD_FINAL']+_item_final['QTD_REDUCAO']
    _mot = (_pf[_pf.get('MOTIVO_AJUSTE_PAI_FILHO',pd.Series(index=_pf.index,dtype=object)).notna()]
        .groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False).agg(MOTIVO_M1=('MOTIVO_AJUSTE_PAI_FILHO','last')))
    _item_final=_item_final.merge(_mot,on=['MES_REF','ID_PROD_UNID_FAT'],how='left')

    if 'ajuste_pai_filho' in globals() and not ajuste_pai_filho.empty:
        _a=ajuste_pai_filho.merge(_item_final,left_on=['MES_REF','ID_PROD_UNID_FAT_ORIGEM'],right_on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
        _a['QTD_PAI_SOLVER']=_num_m1(_a['QTD_INICIAL_EST'])
        _a['QTD_PAI_FINAL_VIAVEL_EST']=_num_m1(_a['QTD_FINAL'])
        _a['FATOR_ATEND_PAI_SOLVER']=np.where(_num_m1(_a['NEC_RAIZ_PCS'])>TOL_FECHAMENTO_M1,_a['QTD_PAI_SOLVER']/_num_m1(_a['NEC_RAIZ_PCS']),0.0)
        _a['FATOR_FINAL_VIAVEL_ORIGEM']=np.where(_num_m1(_a['NEC_RAIZ_PCS'])>TOL_FECHAMENTO_M1,_a['QTD_PAI_FINAL_VIAVEL_EST']/_num_m1(_a['NEC_RAIZ_PCS']),0.0)
        _a['MOTIVO_AJUSTE_PAI_FILHO']=np.where(_num_m1(_a['QTD_REDUCAO'])>TOL_FECHAMENTO_M1,_a['MOTIVO_M1'].fillna('CORTE_PAI_POR_CORTE_SOLVER_COMPONENTE_M1'),_a['MOTIVO_AJUSTE_PAI_FILHO'])
        ajuste_pai_filho=_a[[c for c in ajuste_pai_filho.columns if c in _a.columns]].copy()

    if 'resumo_pai_filho' in globals() and not resumo_pai_filho.empty:
        _r=resumo_pai_filho.merge(_item_final,left_on=['MES_REF','ID_PROD_UNID_FAT_ORIGEM'],right_on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
        _r['QTD_PAI_SOLVER']=_num_m1(_r['QTD_INICIAL_EST'])
        _r['QTD_PAI_FINAL_VIAVEL_EST']=_num_m1(_r['QTD_FINAL'])
        _r['QTD_REDUCAO_POS_PAI_FILHO']=_num_m1(_r['QTD_REDUCAO'])
        _r['FATOR_ATEND_PAI_SOLVER']=np.where(_num_m1(_r['NEC_RAIZ_PCS'])>TOL_FECHAMENTO_M1,_r['QTD_PAI_SOLVER']/_num_m1(_r['NEC_RAIZ_PCS']),0.0)
        _r['FATOR_FINAL_VIAVEL_ORIGEM']=np.where(_num_m1(_r['NEC_RAIZ_PCS'])>TOL_FECHAMENTO_M1,_r['QTD_PAI_FINAL_VIAVEL_EST']/_num_m1(_r['NEC_RAIZ_PCS']),0.0)
        _r['FATOR_AJUSTE_SOBRE_SOLVER']=np.where(_r['QTD_PAI_SOLVER']>TOL_FECHAMENTO_M1,_r['QTD_PAI_FINAL_VIAVEL_EST']/_r['QTD_PAI_SOLVER'],1.0)
        _r['MOTIVO_AJUSTE_PAI_FILHO']=np.where(_r['QTD_REDUCAO_POS_PAI_FILHO']>TOL_FECHAMENTO_M1,_r['MOTIVO_M1'].fillna('CORTE_PAI_POR_CORTE_SOLVER_COMPONENTE_M1'),_r['MOTIVO_AJUSTE_PAI_FILHO'])
        resumo_pai_filho=_r[[c for c in resumo_pai_filho.columns if c in _r.columns]].copy()

    if 'capacidade_liberada_pf' in globals():
        _cap=_pf[_pf['QTD_REDUCAO_POS_PAI_FILHO']>TOL_FECHAMENTO_M1].copy()
        _cap['QTD_PRODUZIR_FINAL_VIAVEL_EST']=_cap['QTD_PRODUZIR_SOLVER']
        _cap['QTD_PRODUZIR_SOLVER']=_cap['QTD_PRODUZIR_SOLVER']+_cap['QTD_REDUCAO_POS_PAI_FILHO']
        _cap['HR_PRODUZIR_SOLVER']=_cap['HR_PRODUZIR_SOLVER']+_cap['HR_LIBERADA_EST']
        _cols=[c for c in capacidade_liberada_pf.columns if c in _cap.columns]
        capacidade_liberada_pf=_cap[_cols].copy()
except Exception as e:
    print('AVISO_M1_RECONCILIACAO_GUIAS_13_14_15:',e)

_realoc_m1_log = pd.concat(logs_realoc_m1, ignore_index=True) if logs_realoc_m1 else pd.DataFrame()
if '_realoc_m1_log' in globals() and not _realoc_m1_log.empty:
    try:
        # Mantém a guia 16 com o layout mais próximo possível, sem criar colunas novas no plano.
        comuns = [c for c in realocacao_cap_liberada.columns if c in _realoc_m1_log.columns]
        if comuns:
            realocacao_cap_liberada = pd.concat([realocacao_cap_liberada, _realoc_m1_log[comuns]], ignore_index=True)
    except Exception:
        pass

# Resumos de capacidade com plano final.
resumo_recursos = (
    plano_producao.groupby(['MES_REF','ALOC_REC'], as_index=False)
    .agg(HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER','sum'), HOR_REC=('HOR_REC','max'))
    if {'MES_REF','ALOC_REC','HR_PRODUZIR_SOLVER','HOR_REC'}.issubset(plano_producao.columns) else resumo_recursos
)
if 'ESTOURO_HR_RECURSO' not in resumo_recursos.columns and {'HR_PRODUZIR_SOLVER','HOR_REC'}.issubset(resumo_recursos.columns):
    resumo_recursos['ESTOURO_HR_RECURSO'] = (resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC']).clip(lower=0.0)
else:
    if {'HR_PRODUZIR_SOLVER','HOR_REC'}.issubset(resumo_recursos.columns):
        resumo_recursos['ESTOURO_HR_RECURSO'] = (resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC']).clip(lower=0.0)

resumo_ferramentas = (
    plano_producao.groupby(['MES_REF','COD_FER_UNID'], as_index=False)
    .agg(HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER','sum'), HOR_FER=('HOR_FER','max'))
    if {'MES_REF','COD_FER_UNID','HR_PRODUZIR_SOLVER','HOR_FER'}.issubset(plano_producao.columns) else resumo_ferramentas
)
if 'ESTOURO_HR_FER' not in resumo_ferramentas.columns and {'HR_PRODUZIR_SOLVER','HOR_FER'}.issubset(resumo_ferramentas.columns):
    resumo_ferramentas['ESTOURO_HR_FER'] = (resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER']).clip(lower=0.0)
else:
    if {'HR_PRODUZIR_SOLVER','HOR_FER'}.issubset(resumo_ferramentas.columns):
        resumo_ferramentas['ESTOURO_HR_FER'] = (resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER']).clip(lower=0.0)

# Ajusta status de itens zerados pela sincronização estrutural final, sem alterar layout.
try:
    _mask_zero_sem_demanda_m1 = (
        (_num_m1(plano_producao['QTD_PRODUZIR_SOLVER']) <= TOL_FECHAMENTO_M1) &
        (_num_m1(plano_producao.get('NEC_PCS_CONSOLIDADA', 0)) <= TOL_FECHAMENTO_M1)
    )
    if 'STATUS_DECISAO' in plano_producao.columns:
        plano_producao.loc[_mask_zero_sem_demanda_m1, 'STATUS_DECISAO'] = 'NAO_PLANEJADO_SEM_DEMANDA'
    if 'MOTIVO_STATUS' in plano_producao.columns:
        plano_producao.loc[_mask_zero_sem_demanda_m1, 'MOTIVO_STATUS'] = 'ITEM_SEM_DEMANDA_NO_MES'
except Exception:
    pass

# Preserva layout de saída validado: não cria, remove ou renomeia colunas nas guias principais.
try:
    resumo_recursos = resumo_recursos.reindex(columns=_COLS_01_RECURSOS_M1)
except Exception:
    pass
try:
    resumo_ferramentas = resumo_ferramentas.reindex(columns=_COLS_02_FERRAMENTAS_M1)
except Exception:
    pass
try:
    plano_producao = plano_producao.reindex(columns=_COLS_03_PLANO_M1)
except Exception:
    pass
try:
    nao_atend_item = nao_atend_item.reindex(columns=_COLS_04_NAO_ATEND_M1)
except Exception:
    pass
try:
    nec_consolidada_solver = nec_consolidada_solver.reindex(columns=_COLS_12_NEC_M1)
except Exception:
    pass

# Validação final direta: plano final dos pais explodido == necessidade final dos filhos.
prod_final_m1 = _prod_item_m1(plano_producao)
arestas_chk_m1, comp_chk_m1 = _estoque_final_por_filho_m1(arestas_base_m1, prod_final_m1, meta_m1)
dem_chk_m1 = _demanda_final_m1(comp_chk_m1, meta_m1)
prod_chk_m1 = prod_final_m1.rename(columns={'QTD_ITEM':'QTD_FILHO_FINAL'})
chk_m1 = dem_chk_m1.merge(prod_chk_m1, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').fillna({'QTD_FILHO_FINAL':0.0})
# Validação estrutural: filho precisa sustentar a NEC_COMP final, considerando estoque consumido e preservando demanda direta.
# Não exige atendimento integral da demanda direta, porque gap direto pode existir por capacidade.
chk_m1['PRODUCAO_UTIL_TOTAL'] = np.minimum(chk_m1['QTD_FILHO_FINAL'].clip(lower=0.0), chk_m1.get('NEC_LIQ_TOTAL', chk_m1['QTD_DEMANDA_FINAL']).clip(lower=0.0))
chk_m1['CORTE_SOLVER_TOTAL'] = (chk_m1.get('NEC_LIQ_TOTAL', chk_m1['QTD_DEMANDA_FINAL']) - chk_m1['PRODUCAO_UTIL_TOTAL']).clip(lower=0.0)
chk_m1['GAP_DIRETO_BLOQUEADO'] = [float(gap_direto_atribuido.get((r.MES_REF,str(r.ID_PROD_UNID_FAT)),0.0)) for r in chk_m1.itertuples(index=False)]
chk_m1['DIF_FILHO_FALTA'] = (chk_m1['CORTE_SOLVER_TOTAL'] - chk_m1['GAP_DIRETO_BLOQUEADO']).clip(lower=0.0)
chk_m1['QTD_FILHO_DISP_ESTRUTURA'] = (chk_m1['NEC_BRUTA_FINAL'] - chk_m1['DIF_FILHO_FALTA']).clip(lower=0.0)
chk_m1['DIF_FILHO_SOBRA'] = 0.0
qtd_falta_m1 = int((chk_m1['DIF_FILHO_FALTA'] > TOL_FECHAMENTO_M1).sum())
qtd_sobra_m1 = 0
rec_estouro_m1 = int((resumo_recursos['ESTOURO_HR_RECURSO'] > TOL_FECHAMENTO_M1).sum()) if 'ESTOURO_HR_RECURSO' in resumo_recursos.columns else 0
# Validacao de pecas inteiras e regras operacionais por item.
_prod_valid_m1 = _prod_item_m1(plano_producao).merge(meta_m1, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
invalid_oper_m1 = 0
for _r in _prod_valid_m1.itertuples(index=False):
    _q=float(_r.QTD_ITEM or 0)
    if abs(_q-round(_q))>1e-9: invalid_oper_m1 += 1; continue
    if _q>TOL_FECHAMENTO_M1:
        _row=_r._asdict(); _cap=_quantidade_valida_abaixo_m1(_q,_row)
        if abs(_cap-_q)>1e-9: invalid_oper_m1 += 1
fer_estouro_m1 = int((resumo_ferramentas['ESTOURO_HR_FER'] > TOL_FECHAMENTO_M1).sum()) if 'ESTOURO_HR_FER' in resumo_ferramentas.columns else 0

print('FECHAMENTO_M1_ITERACOES_EXECUTADAS:', iter_executadas_m1)
print('FECHAMENTO_M1_FILHOS_COM_FALTA:', qtd_falta_m1)
print('FECHAMENTO_M1_FILHOS_COM_SOBRA:', qtd_sobra_m1)
print('FECHAMENTO_M1_RECURSOS_COM_ESTOURO:', rec_estouro_m1)
print('FECHAMENTO_M1_FERRAMENTAS_COM_ESTOURO:', fer_estouro_m1)
print('FECHAMENTO_M1_ITENS_QUANTIDADE_OPERACIONAL_INVALIDA:', invalid_oper_m1)
try:
    sent = chk_m1[chk_m1['ID_PROD_UNID_FAT'].astype(str).str.contains('0497NE', na=False)]
    print('VALIDACAO_M1_0497NE_DEMANDA_FINAL:', float(sent['QTD_DEMANDA_FINAL'].sum()) if not sent.empty else 0.0)
    print('VALIDACAO_M1_0497NE_PRODUCAO_FINAL:', float(sent['QTD_FILHO_FINAL'].sum()) if not sent.empty else 0.0)
except Exception as e:
    print('VALIDACAO_M1_0497NE_FALHOU:', e)
try:
    sent2362 = chk_m1[chk_m1['COD_PROD'].astype(str).eq('2362')]
    print('VALIDACAO_M1_2362_DEMANDA_ESTRUTURAL_BRUTA:', float(sent2362['NEC_BRUTA_FINAL'].sum()) if not sent2362.empty else 0.0)
    print('VALIDACAO_M1_2362_ESTOQUE_APLICADO:', float(sent2362['ESTOQUE_FINAL'].sum()) if not sent2362.empty else 0.0)
    print('VALIDACAO_M1_2362_PRODUCAO_FINAL:', float(sent2362['QTD_FILHO_FINAL'].sum()) if not sent2362.empty else 0.0)
except Exception as e:
    print('VALIDACAO_M1_2362_FALHOU:', e)

if qtd_falta_m1 > 0 or rec_estouro_m1 > 0 or fer_estouro_m1 > 0 or invalid_oper_m1 > 0:
    amostra = chk_m1[(chk_m1['DIF_FILHO_FALTA'] > TOL_FECHAMENTO_M1)].sort_values('DIF_FILHO_FALTA', ascending=False).head(20)
    chk_m1.to_csv('/mnt/data/.csv', index=False)
    try:
        resumo_recursos.to_csv('/mnt/data/.csv', index=False)
        resumo_ferramentas.to_csv('/mnt/data/.csv', index=False)
    except Exception:
        pass
    print('AMOSTRA_ERROS_FECHAMENTO_M1:')
    try:
        display(amostra[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT','NEC_BRUTA_FINAL','ESTOQUE_FINAL','QTD_FILHO_FINAL','QTD_FILHO_DISP_ESTRUTURA','DIF_FILHO_FALTA']])
    except Exception:
        print(amostra.head(20).to_string())
    raise RuntimeError('FECHAMENTO_M1_NAO_VALIDADO: plano final dos pais nao fecha contra filhos finais ou capacidade estourou.')


In [ ]:
# ============================================================
# 07. EXPORTAÇÃO
# ============================================================
# A exportação grava somente as guias operacionais autorizadas.
# Dicionário de guias, explicações de colunas e legenda de status são mantidos na memória oficial do projeto.

# A guia de capacidade liberada é exportada sem fatores auxiliares que não fazem parte da leitura operacional.
capacidade_liberada_pf_export = capacidade_liberada_pf.drop(
    columns=[col for col in ["FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if col in capacidade_liberada_pf.columns]
).copy()

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    resumo_recursos.to_excel(writer, sheet_name="01_RESUMO_RECURSOS", index=False)
    resumo_ferramentas.to_excel(writer, sheet_name="02_RESUMO_FERRAMENTAS", index=False)
    plano_producao.to_excel(writer, sheet_name="03_PLANO_PRODUCAO", index=False)
    nao_atend_item.to_excel(writer, sheet_name="04_NAO_ATEND_ITEM", index=False)
    auditoria_capacidade.to_excel(writer, sheet_name="05_AUDITORIA_CAPACIDADE", index=False)
    alternativas_item.to_excel(writer, sheet_name="06_ALTERNATIVAS_ITEM", index=False)
    rastro_rateio.to_excel(writer, sheet_name="07_RASTRO_RATEIO", index=False)
    diagnostico_item.to_excel(writer, sheet_name="08_DIAGNOSTICO_ITEM", index=False)
    diagnostico_alternativas.to_excel(writer, sheet_name="09_DIAGNOSTICO_ALTERNATIVAS", index=False)
    estrutura_explodida.to_excel(writer, sheet_name="10_ESTRUTURA_EXPLODIDA", index=False)
    nec_explodida_componentes.to_excel(writer, sheet_name="11_NEC_EXPLODIDA_COMPONENTES", index=False)
    nec_consolidada_solver.to_excel(writer, sheet_name="12_NEC_CONSOLIDADA_SOLVER", index=False)
    ajuste_pai_filho.to_excel(writer, sheet_name="13_AJUSTE_PAI_FILHO", index=False)
    resumo_pai_filho.to_excel(writer, sheet_name="14_RESUMO_PAI_FILHO", index=False)
    capacidade_liberada_pf_export.to_excel(writer, sheet_name="15_CAPACIDADE_LIBERADA_PF", index=False)
    realocacao_cap_liberada.to_excel(writer, sheet_name="16_REALOCACAO_CAP_LIBERADA", index=False)
    resumo_item_realocacao.to_excel(writer, sheet_name="18_RESUMO_ITEM_REALOCACAO", index=False)

# Formatação básica para manter leitura consistente das guias exportadas.
wb = load_workbook(OUTPUT_SOLVER_FILE)
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells[:1000]:
            if cell.value is not None:
                max_len = max(max_len, min(len(str(cell.value)), 60))
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)
wb.save(OUTPUT_SOLVER_FILE)


In [ ]:
# 99. CORRECAO FINAL  - CAMINHO DO EXCEL SEM ARQUIVO TEMPORARIO

from pathlib import Path
from openpyxl import load_workbook

# : resolver o caminho real do Excel exportado pela celula anterior.
# Nao criar backup, tmp ou copia intermediaria.
out_path = Path(globals().get("OUTPUT_SOLVER_FILE", globals().get("OUTPUT_SOLVER_FILENAME", "bd_SOLVER_m1.xlsx")))

if not out_path.exists():
    candidatos = [
        Path(globals().get("OUTPUT_SOLVER_FILENAME", "bd_SOLVER_m1.xlsx")),
        Path.cwd() / globals().get("OUTPUT_SOLVER_FILENAME", "bd_SOLVER_m1.xlsx"),
        Path("/mnt/data") / globals().get("OUTPUT_SOLVER_FILENAME", "bd_SOLVER_m1.xlsx"),
    ]
    achados = [p for p in candidatos if p.exists()]
    if achados:
        out_path = achados[0]
    else:
        raise FileNotFoundError(
            f"Excel exportado nao encontrado. Esperado em OUTPUT_SOLVER_FILE={out_path}. "
            f"Candidatos verificados: {[str(p) for p in candidatos]}"
        )

# Abre e salva diretamente o Excel final. Nada de arquivo temporario.
wb = load_workbook(out_path)
wb.save(out_path)


In [ ]:
# ============================================================
# 09. TEMPO TOTAL DE EXECUÇÃO
# ============================================================
# O temporizador é encerrado somente depois de todo o processamento e da gravação final do Excel.
timer.finalizar()
